In [ ]:
# Experiment settings fixed before strict unseen test evaluation


NOTEBOOK_VERSION = "v7.4-corpus-association-decision-rule"
VALIDATION_DESIGN = "single_researcher_interim"
INTERIM_RESOURCE = True
SEED = 42

TEXT_COL = "SN(Original Shona Tweet)"
LABEL_COL = "finalLabel3Classes"

# Candidate evidence
MIN_DOC_SUPPORT = 10
BOOTSTRAP_REPS = 2000
CI_LEVEL = 0.95
FDR_ALPHA = 0.05
MIN_WILSON_LOWER = 0.50

# English/code-switch filtering
USE_WORDFREQ_ENGLISH_FILTER = True
REQUIRE_WORDFREQ = True
ENGLISH_ZIPF_THRESHOLD = 2.5

# Single-researcher interim validation
RESEARCHER_REVIEWER_ID = "RESEARCHER"
MIN_RESEARCHER_REVIEWERS = 1
MIN_VALID_SHONA_SHARE = 1.0
MIN_POLARITY_SHARE = 1.0
MAX_NON_CONTENT_SHARE = 0.5
REQUIRE_WORD_TYPE = True
REQUIRE_COMPLETE_RESEARCHER_REVIEW = True
REQUIRE_RESEARCHER_MEANING = True
REJECT_IDENTITY_TERMS_IN_INTERIM = True
ELIGIBLE_SHONA_LEVELS = {"NATIVE", "L1", "FIRST_LANGUAGE", "MOTHER_TONGUE", "FLUENT"}
RESEARCHER_REVIEW_REQUIRED_FIELDS = (
    "reviewer_id", "candidate_id", "word", "shona_level", "valid_shona",
    "meaning", "sentiment", "confidence", "word_type"
)
RESEARCHER_REVIEW_OPTIONAL_FIELDS = ("comment",)
EXPECTED_SENTIMENT_LABELS = ("NEG", "NEU", "POS")

# RAG/GPT evidence
USE_RAG_GPT = True
OPENAI_SECRET_NAME = "OPENAI_API_KEY"
RAG_MODEL = "gpt-5-mini"
RAG_MAX_CONTEXTS_PER_TOKEN = 6
RAG_SLEEP_SECONDS = 0.35
RAG_MAX_RETRIES = 5
RAG_RECOVERY_NOISE_REASONS = {"english_frequency"}
RAG_RECOVERY_MIN_DOCUMENT_SUPPORT = MIN_DOC_SUPPORT
RAG_RECOVERY_BATCH_SIZE = 12
RAG_CONTEXT_BATCH_SIZE = 8
RAG_CONTEXT_COMPLETION_RETRIES = 3
RAG_RECOVERY_MIN_CONFIDENCE = 0.70
RAG_RECOVERY_ALLOWED_CLASSES = {"shona_content"}
RAG_CONTEXT_MIN_CONFIDENCE = 0.70
FINAL_DECISION_RULE_VERSION = "v7.4-lexical-polarity-vs-corpus-association"
CONTEXTUAL_STATUSES_ALLOWED_FOR_CORPUS_ASSOCIATION = {"support", "review"}
RAG_CACHE_DIRECTORY = "/kaggle/working/FrenchyShona_V2_RAG_Cache_v7_2"
RAG_RECOVERY_PROMPT_VERSION = "candidate_language_screen_v1"
RAG_CONTEXT_PROMPT_VERSION = "candidate_context_assessment_v2_complete"

# Data and resource safeguards
EXCLUDE_LABEL_CONFLICTS = True
EXPECTED_INPUT_BASE_LEXICON_ROWS = 6632
EXPECTED_QUARANTINED_SCORE_ROWS = 17
EXPECTED_EVALUATION_BASE_LEXICON_ROWS = 6615
EXPECTED_QUARANTINE_SIGNATURE_SHA256 = "5b6bbf420139890c31b2183b7381b13415c566a2a96fd1dfc038bffb16720114"
EXPECTED_CLEAN_BASE_CORE_SHA256 = "213d4728d5f14f69e897f4448f53f6630a811379d1884e84ce22a1784d148098"
BASE_SCORE_MIN = -9
BASE_SCORE_MAX = 9
ENFORCE_INPUT_BASE_LEXICON_ROWS = True
ENFORCE_QUARANTINE_COUNT = True
ENFORCE_QUARANTINE_SIGNATURE = True
ENFORCE_CLEAN_BASE_ROWS = True
ENFORCE_CLEAN_BASE_MARKERS = True

# Development split
VALIDATION_FRACTION = 0.10

# Evaluation
THRESHOLD_GRID_STEP = 0.5
RUN_CLASSICAL_BASELINES = True
LEXICON_FORM_SCORE_AGGREGATION = "median"
PROBABILITY_SUM_TOLERANCE = 1e-5
REQUIRE_MODEL_PROVENANCE = True
MODEL_ALPHA_GRID = [round(x, 2) for x in __import__("numpy").arange(0.0, 1.0001, 0.05)]

# Optional local testing override. Leave as None on Kaggle.
INPUT_ROOT_OVERRIDE = None
OUTPUT_ROOT = "/kaggle/working/FrenchyShona_V2_RAG_Complete_V7_4"
RESET_OUTPUT_ROOT = True


In [ ]:

# Imports, package checks and shared helpers

import os
import re
import json
import math
import time
import hashlib
import shutil
import zipfile
import unicodedata
import subprocess
import sys
import importlib
import platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn

from scipy.stats import fisher_exact, norm
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

np.random.seed(SEED)

def ensure_package(import_name, pip_spec):
    try:
        return importlib.import_module(import_name)
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_spec])
            return importlib.import_module(import_name)
        except Exception as exc:
            raise RuntimeError(
                f"Required package {import_name!r} could not be loaded. "
                "Install the required package or enable internet access in the execution environment."
            ) from exc

# Excel input is required by the workflow.
openpyxl = ensure_package("openpyxl", "openpyxl>=3.1")
openai_pkg = ensure_package("openai", "openai>=1.40")

# wordfreq is required by the English/code-switch filtering procedure.
# The workflow stops if the required filter cannot be reproduced.
WORD_FREQ_AVAILABLE = False
zipf_frequency = None
if USE_WORDFREQ_ENGLISH_FILTER:
    try:
        from wordfreq import zipf_frequency
        WORD_FREQ_AVAILABLE = True
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wordfreq==3.1.1"])
            from wordfreq import zipf_frequency
            WORD_FREQ_AVAILABLE = True
        except Exception as exc:
            if REQUIRE_WORDFREQ:
                raise RuntimeError(
                    "wordfreq could not be loaded. Install the required package before execution. "
                    "The workflow will not continue with a different English-frequency filter."
                ) from exc
            print("WARNING: wordfreq unavailable; frequency filtering is inactive:", exc)

def select_output_root(configured_root):
    configured = Path(configured_root).expanduser()
    candidates = [
        ("configured OUTPUT_ROOT", configured),
        ("Kaggle working fallback", Path("/kaggle/working/FrenchyShona_V2_RAG_Complete_V7_4")),
        ("current-directory fallback", Path.cwd() / "FrenchyShona_V2_RAG_Complete_V7_4"),
        ("mnt-data fallback", Path("/mnt/data/FrenchyShona_V2_RAG_Complete_V7_4")),
        ("home-directory fallback", Path.home() / "FrenchyShona_V2_RAG_Complete_V7_4"),
    ]

    errors = []
    seen = set()
    for label, candidate in candidates:
        key = str(candidate.resolve(strict=False))
        if key in seen:
            continue
        seen.add(key)
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            if candidate != configured:
                print(
                    f"WARNING: OUTPUT_ROOT {configured} could not be used. "
                    f"Using {candidate} ({label})."
                )
            else:
                print(f"Using configured OUTPUT_ROOT: {candidate}")
            return candidate
        except Exception as exc:
            errors.append(f"{label}: {candidate}: {type(exc).__name__}: {exc}")

    raise RuntimeError(
        "No writable output directory was available. Tried:\n- " + "\n- ".join(errors)
    )


ROOT_OUT = select_output_root(OUTPUT_ROOT)

# Prevent stale frozen files or test metrics from surviving a failed prior run.
if RESET_OUTPUT_ROOT:
    resolved_root = ROOT_OUT.resolve(strict=False)
    unsafe_roots = {
        Path("/").resolve(strict=False),
        Path.home().resolve(strict=False),
        Path.cwd().resolve(strict=False),
        Path("/kaggle/working").resolve(strict=False),
        Path("/mnt/data").resolve(strict=False),
    }
    if resolved_root in unsafe_roots:
        raise RuntimeError(
            f"Refusing to reset unsafe OUTPUT_ROOT: {ROOT_OUT}. "
            "Use a dedicated subfolder such as .../FrenchyShona_V2_RAG_Complete_V7_4."
        )
    if ROOT_OUT.exists():
        shutil.rmtree(ROOT_OUT)
    ROOT_OUT.mkdir(parents=True, exist_ok=True)
    print("Reset output folder for a clean run:", ROOT_OUT)

# Remove the previous canonical archive at the beginning. If this run fails, an old ZIP
# cannot be mistaken for the current results. The final cell writes a temporary ZIP and
# atomically renames it only after the archive closes successfully.
CANONICAL_RESULTS_ZIP = ROOT_OUT.parent / "FrenchyShona_V2_RAG_Complete_V7_4_results.zip"
if CANONICAL_RESULTS_ZIP.exists():
    CANONICAL_RESULTS_ZIP.unlink()
    print("Removed stale results ZIP:", CANONICAL_RESULTS_ZIP)

FIG = ROOT_OUT / "figures"
TAB = ROOT_OUT / "tables"
AUD = ROOT_OUT / "audit"
REV = ROOT_OUT / "single_researcher_review"
EVAL = ROOT_OUT / "evaluation"
MODEL = ROOT_OUT / "optional_model_hybrid"
for folder in [ROOT_OUT, FIG, TAB, AUD, REV, EVAL, MODEL]:
    folder.mkdir(parents=True, exist_ok=True)

figure_manifest = []


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_lexical_text(value):
    text = normalize_text(value)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    # Keep the lexical content of hashtags while removing the marker itself.
    text = re.sub(r"#([a-zA-Z]+)", r" \1 ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-Z'\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_repeats(word, max_repeats=2):
    return re.sub(r"(.)\1{" + str(max_repeats) + r",}", lambda m: m.group(1) * max_repeats, str(word))


def tokenize_lexical_text(value):
    return [
        normalize_repeats(word, 2)
        for word in clean_lexical_text(value).split()
        if isinstance(word, str) and word
    ]


def norm_token(value):
    return normalize_repeats(normalize_text(value).replace(" ", ""), 2)


def norm_label(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        text = str(int(value))
    elif isinstance(value, (float, np.floating)) and float(value).is_integer():
        text = str(int(value))
    else:
        text = str(value).strip().upper()
    mapping = {
        "POSITIVE": "POS", "NEGATIVE": "NEG", "NEUTRAL": "NEU",
        "0": "NEG", "1": "NEU", "2": "POS"
    }
    return mapping.get(text, text)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value):
    """Convert numpy values and non-finite floats to portable JSON values."""
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(json_safe(payload), handle, indent=2, allow_nan=False)


def json_text(payload):
    return json.dumps(json_safe(payload), indent=2, allow_nan=False)


def save_current_figure(names, dpi=300):
    if isinstance(names, str):
        names = [names]
    for name in names:
        path = FIG / name
        plt.savefig(path, dpi=dpi, bbox_inches="tight")
        figure_manifest.append({"file": name, "status": "generated"})
    plt.close()


def mark_figure_pending(name, reason):
    figure_manifest.append({"file": name, "status": "pending", "reason": reason})


def save_table_image(df, title, filename, max_rows=15, fontsize=9):
    view = df.head(max_rows).copy()
    fig, ax = plt.subplots(figsize=(12, max(3.0, 0.38 * (len(view) + 2))))
    ax.axis("off")
    ax.set_title(title)
    table = ax.table(
        cellText=view.astype(str).values,
        colLabels=view.columns.astype(str),
        cellLoc="center", colLoc="center", loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.35)
    save_current_figure(filename)


def all_input_files():
    roots = []
    if INPUT_ROOT_OVERRIDE:
        roots.append(Path(INPUT_ROOT_OVERRIDE))
    elif Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    elif Path("/mnt/data").exists():
        roots.append(Path("/mnt/data"))
    files = []
    for root in roots:
        files.extend([p for p in root.rglob("*") if p.is_file()])
    return files


INPUT_FILES = all_input_files()


def find_inputs_by_name(names):
    if isinstance(names, str):
        names = [names]
    wanted = {name.casefold() for name in names}
    return [p for p in INPUT_FILES if p.name.casefold() in wanted]


def find_input(names, required=True, prefer_terms=()):
    matches = find_inputs_by_name(names)
    if matches and prefer_terms:
        matches = sorted(
            matches,
            key=lambda p: (-sum(term.casefold() in str(p).casefold() for term in prefer_terms), len(str(p)))
        )
    if matches:
        return matches[0]
    if required:
        label = ", ".join([names] if isinstance(names, str) else names)
        raise FileNotFoundError(f"Required input not found: {label}")
    return None


def find_unique_input(names, role, required=True, prefer_terms=()):
    matches = find_inputs_by_name(names)
    if not matches:
        if required:
            label = ", ".join([names] if isinstance(names, str) else names)
            raise FileNotFoundError(f"Required {role} input not found: {label}")
        return None
    records = [{"role": role, "path": str(path), "sha256": sha256_file(path)} for path in matches]
    pd.DataFrame(records).to_csv(AUD / f"{re.sub(r'[^a-z0-9]+', '_', role.casefold()).strip('_')}_input_candidates.csv", index=False)
    distinct_hashes = {record["sha256"] for record in records}
    if len(distinct_hashes) > 1:
        raise ValueError(
            f"Multiple different files were found for {role}. Remove the ambiguous copies; "
            f"see audit/{re.sub(r'[^a-z0-9]+', '_', role.casefold()).strip('_')}_input_candidates.csv."
        )
    if prefer_terms:
        matches = sorted(
            matches,
            key=lambda path: (
                -sum(term.casefold() in str(path).casefold() for term in prefer_terms),
                len(str(path)),
                str(path),
            ),
        )
    else:
        matches = sorted(matches, key=lambda path: (len(str(path)), str(path)))
    return matches[0]


def read_any(path):
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")


def normalise_column_name(value):
    return re.sub(r"\s+", " ", str(value).strip()).casefold()


def find_col_ci(df, names):
    lookup = {normalise_column_name(col): col for col in df.columns}
    for name in names:
        key = normalise_column_name(name)
        if key in lookup:
            return lookup[key]
    return None


def copy_alias(source_path, alias_name):
    source_path = Path(source_path)
    if source_path.exists():
        shutil.copy2(source_path, FIG / alias_name)
        figure_manifest.append({"file": alias_name, "status": "generated_alias"})


def benjamini_hochberg(pvalues, alpha):
    p = np.asarray(pvalues, dtype=float)
    m = len(p)
    if m == 0:
        return np.zeros(0, dtype=bool), np.zeros(0, dtype=float)
    order = np.argsort(p)
    ranked = p[order]
    adjusted_ranked = np.minimum.accumulate((ranked * m / np.arange(1, m + 1))[::-1])[::-1]
    adjusted_ranked = np.clip(adjusted_ranked, 0.0, 1.0)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_ranked
    return adjusted <= alpha, adjusted


def wilson_interval(successes, total, confidence=0.95):
    if total <= 0:
        return np.nan, np.nan
    z = float(norm.ppf(1 - (1 - confidence) / 2))
    proportion = successes / total
    denominator = 1 + (z * z) / total
    centre = (proportion + (z * z) / (2 * total)) / denominator
    half = z * math.sqrt((proportion * (1 - proportion) / total) + (z * z) / (4 * total * total)) / denominator
    return centre - half, centre + half


def macro_f1_numpy(gold_matrix, pred_matrix):
    result = np.zeros(gold_matrix.shape[0], dtype=float)
    for label in range(3):
        tp = ((gold_matrix == label) & (pred_matrix == label)).sum(axis=1)
        fp = ((gold_matrix != label) & (pred_matrix == label)).sum(axis=1)
        fn = ((gold_matrix == label) & (pred_matrix != label)).sum(axis=1)
        denominator = 2 * tp + fp + fn
        class_f1 = np.divide(2 * tp, denominator, out=np.zeros_like(denominator, dtype=float), where=denominator != 0)
        result += class_f1 / 3.0
    return result


def paired_bootstrap_macro_f1(gold, pred_a, pred_b, reps=2000, seed=42):
    label_to_int = {"NEG": 0, "NEU": 1, "POS": 2}
    gold = np.asarray(gold, dtype=object)
    pred_a = np.asarray(pred_a, dtype=object)
    pred_b = np.asarray(pred_b, dtype=object)
    if not (len(gold) == len(pred_a) == len(pred_b)) or len(gold) == 0:
        raise ValueError("Paired bootstrap inputs must be non-empty and have equal length.")
    for name, values in [("gold", gold), ("pred_a", pred_a), ("pred_b", pred_b)]:
        invalid = sorted(set(values) - set(label_to_int))
        if invalid:
            raise ValueError(f"Invalid labels in {name}: {invalid}")

    g = np.asarray([label_to_int[x] for x in gold], dtype=np.int8)
    a = np.asarray([label_to_int[x] for x in pred_a], dtype=np.int8)
    b = np.asarray([label_to_int[x] for x in pred_b], dtype=np.int8)
    observed_a = float(f1_score(gold, pred_a, labels=["NEG", "NEU", "POS"], average="macro", zero_division=0))
    observed_b = float(f1_score(gold, pred_b, labels=["NEG", "NEU", "POS"], average="macro", zero_division=0))

    rng = np.random.default_rng(seed)
    delta_parts, a_parts, b_parts = [], [], []
    batch = 250
    done = 0
    while done < reps:
        size = min(batch, reps - done)
        idx = rng.integers(0, len(g), size=(size, len(g)))
        gg, aa, bb = g[idx], a[idx], b[idx]
        fa = macro_f1_numpy(gg, aa)
        fb = macro_f1_numpy(gg, bb)
        a_parts.append(fa)
        b_parts.append(fb)
        delta_parts.append(fa - fb)
        done += size

    a_boot = np.concatenate(a_parts)
    b_boot = np.concatenate(b_parts)
    delta = np.concatenate(delta_parts)
    q_low, q_high = (1 - CI_LEVEL) / 2, 1 - (1 - CI_LEVEL) / 2
    delta_low, delta_high = np.quantile(delta, [q_low, q_high])
    return {
        "observed_a_macro_f1": observed_a,
        "observed_b_macro_f1": observed_b,
        "observed_delta_a_minus_b": observed_a - observed_b,
        "a_macro_f1_ci_low": float(np.quantile(a_boot, q_low)),
        "a_macro_f1_ci_high": float(np.quantile(a_boot, q_high)),
        "b_macro_f1_ci_low": float(np.quantile(b_boot, q_low)),
        "b_macro_f1_ci_high": float(np.quantile(b_boot, q_high)),
        "delta_macro_f1_ci_low": float(delta_low),
        "delta_macro_f1_ci_high": float(delta_high),
        "delta_ci_excludes_zero": bool(delta_low > 0 or delta_high < 0),
    }


print("Output folder:", ROOT_OUT)
print("wordfreq active:", WORD_FREQ_AVAILABLE)


In [ ]:

# Locate inputs and select the 6,632-row pre-addition source lexicon automatically

TRAIN_PATH = find_unique_input("traindata1.2.xlsx", "ShonaSenti train", prefer_terms=("shonasenti",))
TEST_PATH = find_unique_input("testdata1.2.xlsx", "ShonaSenti test", prefer_terms=("shonasenti",))

BASE_LEXICON_NAMES = [
    "a2_expanded_shona_annotated.csv",
    "final_expanded_shona_annotated.csv",
    "a2_with_shona_gt (1).csv",
    "a2_with_shona_gt.csv",
    "expanded_lexicon_cleaned.xlsx",
    "expanded_lexicon.xlsx",
]

base_audit_rows = []
valid_base_candidates = []
for path in find_inputs_by_name(BASE_LEXICON_NAMES):
    try:
        frame = read_any(path)
        required_cols = {
            "shona": find_col_ci(frame, ["Shona"]),
            "expanded_shona": find_col_ci(frame, ["expanded_shona"]),
            "score": find_col_ci(frame, ["Score"]),
        }
        valid_schema = all(required_cols.values())
        core_digest = None
        if valid_schema:
            core = frame[[required_cols["shona"], required_cols["expanded_shona"], required_cols["score"]]].copy()
            core.columns = ["Shona", "expanded_shona", "Score"]
            core["Shona"] = core["Shona"].astype("string").fillna("").map(normalize_text)
            core["expanded_shona"] = core["expanded_shona"].astype("string").fillna("").map(normalize_text)
            numeric_core_scores = pd.to_numeric(core["Score"], errors="coerce")
            core["Score"] = [
                "" if pd.isna(value) else format(float(value), ".12g")
                for value in numeric_core_scores
            ]
            core = core.sort_values(["Shona", "expanded_shona", "Score"], kind="mergesort").reset_index(drop=True)
            payload = core.to_csv(index=False, lineterminator="\n").encode("utf-8")
            core_digest = hashlib.sha256(payload).hexdigest()
        base_audit_rows.append({
            "path": str(path), "rows": len(frame), "columns": frame.shape[1],
            "valid_schema": valid_schema, "sha256": sha256_file(path),
            "core_shona_expanded_score_sha256": core_digest,
        })
        if valid_schema and len(frame) == EXPECTED_INPUT_BASE_LEXICON_ROWS:
            valid_base_candidates.append((path, core_digest))
    except Exception as exc:
        base_audit_rows.append({"path": str(path), "error": str(exc)})

base_audit = pd.DataFrame(base_audit_rows)
base_audit.to_csv(AUD / "base_lexicon_candidate_files.csv", index=False)

if not valid_base_candidates:
    raise ValueError(
        "No valid 6,632-row pre-addition source lexicon was found. "
        "Review audit/base_lexicon_candidate_files.csv before continuing."
    )

# Multiple copies are acceptable only when the core Shona/expanded_shona/Score content agrees.
core_digests = {digest for _, digest in valid_base_candidates}
if len(core_digests) > 1:
    raise ValueError(
        "Multiple 6,632-row base lexicons were found with different core content. "
        "Choose one authoritative input and remove the others; see audit/base_lexicon_candidate_files.csv."
    )

# Prefer the annotated CSV when identical core copies are present.
valid_base_candidates = sorted(
    [path for path, _ in valid_base_candidates],
    key=lambda p: (
        0 if "expanded_shona_annotated" in p.name.casefold() else 1,
        0 if p.suffix.casefold() == ".csv" else 1,
        len(str(p))
    )
)
MASTER_LEX_PATH = valid_base_candidates[0]

A2_GT_PATH = find_input(
    ["a2_with_shona_gt (1).csv", "a2_with_shona_gt.csv"],
    required=False, prefer_terms=("lexicon",)
)
ORIGINAL_LEX_PATH = find_input("lexicon_6000 words.xlsx", required=False)
WORKING_EXPANDED_XLSX = find_input(
    ["expanded_lexicon.xlsx", "expanded_lexicon_cleaned.xlsx"], required=False
)
# This workflow reproduces the single-researcher review used in the reported experiment.
multi_reviewer_files = find_inputs_by_name(
    ["native_speaker_reviews.csv", "frenchyshona_native_speaker_reviews.csv"]
)
if multi_reviewer_files:
    raise ValueError(
        "This workflow reproduces the single-researcher review used in the reported experiment. "
        "Remove multi-reviewer review files from this input set before continuing."
    )

REVIEW_PATH = find_unique_input(
    ["single_researcher_review.csv", "frenchyshona_single_researcher_review.csv",
     "single_researcher_review_template.csv", "single_researcher_review_template (1).csv"],
    "single-researcher interim review", required=False,
    prefer_terms=("single", "researcher", "review")
)

# Optional contextual-model probability files. These are not required for the lexicon rebuild.
DEV_MODEL_PROBS_PATH = find_unique_input(
    ["validation_model_probabilities.csv", "strict_validation_model_probabilities.csv"],
    "validation model probabilities", required=False
)
TEST_MODEL_PROBS_PATH = find_unique_input(
    ["strict_test_model_probabilities.csv"], "strict-test model probabilities", required=False
)
MODEL_PROVENANCE_PATH = find_unique_input(
    ["model_probability_provenance.json"], "model probability provenance", required=False
)

optional_probability_inputs = [DEV_MODEL_PROBS_PATH, TEST_MODEL_PROBS_PATH]
if any(optional_probability_inputs) and not all(optional_probability_inputs):
    raise ValueError(
        "Only one contextual-model probability file was found. Supply both validation and "
        "strict-test probability files, or remove the incomplete optional input."
    )
if MODEL_PROVENANCE_PATH and not all(optional_probability_inputs):
    raise ValueError(
        "model_probability_provenance.json was supplied without both probability files."
    )

print("Train:", TRAIN_PATH)
print("Test:", TEST_PATH)
print("Selected base lexicon:", MASTER_LEX_PATH)
print("Single-researcher interim review:", REVIEW_PATH if REVIEW_PATH else "not supplied")
print("Optional model probabilities:", bool(DEV_MODEL_PROBS_PATH and TEST_MODEL_PROBS_PATH))

input_records = []
for role, path in [
    ("ShonaSenti train", TRAIN_PATH), ("ShonaSenti test", TEST_PATH),
    ("6,632-row source lexicon", MASTER_LEX_PATH), ("Shona_gt audit", A2_GT_PATH),
    ("original FrenchyLuba", ORIGINAL_LEX_PATH), ("expanded lexicon xlsx", WORKING_EXPANDED_XLSX),
    ("single-researcher interim review", REVIEW_PATH), ("validation model probabilities", DEV_MODEL_PROBS_PATH),
    ("test model probabilities", TEST_MODEL_PROBS_PATH),
    ("model probability provenance", MODEL_PROVENANCE_PATH),
]:
    if path:
        input_records.append({"role": role, "path": str(path), "sha256": sha256_file(path)})
pd.DataFrame(input_records).to_csv(AUD / "input_file_checksums.csv", index=False)


In [ ]:

# Create the duplicate-safe training and strict unseen test boundary

train_raw = pd.read_excel(TRAIN_PATH)
test_raw = pd.read_excel(TEST_PATH)
original_train_rows = int(len(train_raw))
original_test_rows = int(len(test_raw))
train_raw = train_raw.copy()
test_raw = test_raw.copy()
train_raw["_source_partition"] = "published_train"
test_raw["_source_partition"] = "published_test"
train_raw["_source_row_number"] = np.arange(2, len(train_raw) + 2)
test_raw["_source_row_number"] = np.arange(2, len(test_raw) + 2)

required_columns = {TEXT_COL, LABEL_COL}
for name, frame in [("train", train_raw), ("test", test_raw)]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

train_raw["_norm_text"] = train_raw[TEXT_COL].map(normalize_text)
test_raw["_norm_text"] = test_raw[TEXT_COL].map(normalize_text)
train_raw["_lexical_norm_text"] = train_raw[TEXT_COL].map(clean_lexical_text)
test_raw["_lexical_norm_text"] = test_raw[TEXT_COL].map(clean_lexical_text)
train_raw["_label"] = train_raw[LABEL_COL].map(norm_label)
test_raw["_label"] = test_raw[LABEL_COL].map(norm_label)

valid_label_set = set(EXPECTED_SENTIMENT_LABELS)
train_invalid_label_mask = ~train_raw["_label"].isin(valid_label_set)
test_invalid_label_mask = ~test_raw["_label"].isin(valid_label_set)

invalid_label_columns = [
    "_source_partition", "_source_row_number", TEXT_COL, LABEL_COL, "_label"
]
invalid_label_rows = pd.concat([
    train_raw.loc[train_invalid_label_mask, invalid_label_columns],
    test_raw.loc[test_invalid_label_mask, invalid_label_columns],
], ignore_index=True)
invalid_label_rows["invalid_reason"] = "unrecognised_sentiment_label"
invalid_label_rows.to_csv(AUD / "invalid_shonasenti_label_rows.csv", index=False)

label_encoding_audit = {
    "expected_labels": list(EXPECTED_SENTIMENT_LABELS),
    "published_train_rows": original_train_rows,
    "published_test_rows": original_test_rows,
    "recognised_train_labels": int((~train_invalid_label_mask).sum()),
    "recognised_test_labels": int((~test_invalid_label_mask).sum()),
    "unrecognised_train_labels": int(train_invalid_label_mask.sum()),
    "unrecognised_test_labels": int(test_invalid_label_mask.sum()),
    "status": "pass" if len(invalid_label_rows) == 0 else "fail",
}
write_json(AUD / "shonasenti_label_encoding_audit.json", label_encoding_audit)

print(
    f"Recognised train labels: {label_encoding_audit['recognised_train_labels']} / "
    f"{label_encoding_audit['published_train_rows']}"
)
print(
    f"Recognised test labels: {label_encoding_audit['recognised_test_labels']} / "
    f"{label_encoding_audit['published_test_rows']}"
)

if len(invalid_label_rows):
    raise ValueError(
        "Unrecognised ShonaSenti labels were found. No rows are silently dropped. "
        "Inspect audit/invalid_shonasenti_label_rows.csv and correct the input or label mapping."
    )

train_blank_text_mask = train_raw["_norm_text"].eq("")
test_blank_text_mask = test_raw["_norm_text"].eq("")
blank_text_rows = pd.concat([
    train_raw.loc[train_blank_text_mask, invalid_label_columns],
    test_raw.loc[test_blank_text_mask, invalid_label_columns],
], ignore_index=True)
blank_text_rows["invalid_reason"] = "blank_normalised_text"
blank_text_rows.to_csv(AUD / "blank_shonasenti_text_rows.csv", index=False)

train_raw = train_raw.loc[~train_blank_text_mask].copy()
test_raw = test_raw.loc[~test_blank_text_mask].copy()

train_conflicts = set(train_raw.groupby("_norm_text")["_label"].nunique().loc[lambda s: s > 1].index)
test_conflicts_all = set(test_raw.groupby("_norm_text")["_label"].nunique().loc[lambda s: s > 1].index)

train_unique = train_raw.drop_duplicates("_norm_text", keep="first").copy()
train_text_set = set(train_raw["_norm_text"])
test_without_train_overlap = test_raw[~test_raw["_norm_text"].isin(train_text_set)].copy()
test_unseen_unique = test_without_train_overlap.drop_duplicates("_norm_text", keep="first").copy()
test_conflicts_unseen = set(
    test_without_train_overlap.groupby("_norm_text")["_label"].nunique().loc[lambda s: s > 1].index
)

if EXCLUDE_LABEL_CONFLICTS:
    train_strict = train_unique[~train_unique["_norm_text"].isin(train_conflicts)].copy()
    test_strict = test_unseen_unique[~test_unseen_unique["_norm_text"].isin(test_conflicts_unseen)].copy()
else:
    train_strict = train_unique.copy()
    test_strict = test_unseen_unique.copy()

if not train_strict["_norm_text"].is_unique or not test_strict["_norm_text"].is_unique:
    raise AssertionError("Strict development and test partitions must contain unique normalised texts.")
if not set(train_strict["_norm_text"]).isdisjoint(set(test_strict["_norm_text"])):
    raise AssertionError("Strict development and test partitions still overlap.")

# The resource is built from the build split only. Validation is reserved for threshold
# selection and the strict test is reserved for final evaluation.
strict_class_counts = train_strict["_label"].value_counts().reindex(["NEG", "NEU", "POS"], fill_value=0)
if (strict_class_counts < 2).any():
    raise ValueError(
        "The strict development pool does not contain enough rows in every class for a "
        f"stratified build/validation split: {strict_class_counts.to_dict()}"
    )
if set(test_strict["_label"]) != {"NEG", "NEU", "POS"}:
    raise ValueError(
        "The strict unseen test must contain all three classes; observed "
        f"{test_strict['_label'].value_counts().to_dict()}"
    )

build_idx, validation_idx = train_test_split(
    np.arange(len(train_strict)),
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=train_strict["_label"].to_numpy()
)
train_build = train_strict.iloc[np.sort(build_idx)].copy()
train_validation = train_strict.iloc[np.sort(validation_idx)].copy()

def frame_text_label_sha256(frame):
    pairs = sorted(
        (str(text), str(label))
        for text, label in zip(frame["_norm_text"], frame["_label"])
    )
    payload = json.dumps(pairs, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

BOUNDARY_BUILD_TEXTS = frozenset(train_build["_norm_text"])
BOUNDARY_VALIDATION_TEXTS = frozenset(train_validation["_norm_text"])
BOUNDARY_TEST_TEXTS = frozenset(test_strict["_norm_text"])
BOUNDARY_BUILD_ROWS = int(len(train_build))
BOUNDARY_BUILD_DIGEST = frame_text_label_sha256(train_build)
BOUNDARY_VALIDATION_DIGEST = frame_text_label_sha256(train_validation)
BOUNDARY_TEST_DIGEST = frame_text_label_sha256(test_strict)

partition_overlap_counts = {
    "build_validation": len(BOUNDARY_BUILD_TEXTS & BOUNDARY_VALIDATION_TEXTS),
    "build_test": len(BOUNDARY_BUILD_TEXTS & BOUNDARY_TEST_TEXTS),
    "validation_test": len(BOUNDARY_VALIDATION_TEXTS & BOUNDARY_TEST_TEXTS),
}
if any(partition_overlap_counts.values()):
    raise AssertionError(f"Partition boundary overlap detected: {partition_overlap_counts}")
if len(train_build) + len(train_validation) != len(train_strict):
    raise AssertionError("Build and validation rows do not reconstruct the strict development pool.")

write_json(AUD / "partition_boundary_manifest.json", {
    "build_rows": BOUNDARY_BUILD_ROWS,
    "validation_rows": int(len(train_validation)),
    "strict_test_rows": int(len(test_strict)),
    "build_text_label_sha256": BOUNDARY_BUILD_DIGEST,
    "validation_text_label_sha256": BOUNDARY_VALIDATION_DIGEST,
    "strict_test_text_label_sha256": BOUNDARY_TEST_DIGEST,
    "build_validation_overlap": partition_overlap_counts["build_validation"],
    "build_test_overlap": partition_overlap_counts["build_test"],
    "validation_test_overlap": partition_overlap_counts["validation_test"],
    "status": "pass",
})

overlap_exact = set(train_raw["_norm_text"]) & set(test_raw["_norm_text"])
train_lexical_nonblank = set(train_raw.loc[train_raw["_lexical_norm_text"].ne(""), "_lexical_norm_text"])
test_lexical_nonblank = set(test_raw.loc[test_raw["_lexical_norm_text"].ne(""), "_lexical_norm_text"])
overlap_lexical = train_lexical_nonblank & test_lexical_nonblank
overlap_test_rows = int(test_raw["_norm_text"].isin(overlap_exact).sum())

summary = pd.DataFrame([
    ["published_train_rows_before_validity_filter", original_train_rows],
    ["published_test_rows_before_validity_filter", original_test_rows],
    ["published_train_rows_after_validity_filter", len(train_raw)],
    ["published_test_rows_after_validity_filter", len(test_raw)],
    ["unique_train_texts", train_raw["_norm_text"].nunique()],
    ["unique_test_texts", test_raw["_norm_text"].nunique()],
    ["unique_exact_train_test_overlap_texts", len(overlap_exact)],
    ["unique_lexically_normalised_overlap_texts_sensitivity", len(overlap_lexical)],
    ["test_rows_also_seen_in_train", overlap_test_rows],
    ["test_overlap_percent", round(100 * overlap_test_rows / len(test_raw), 4)],
    ["train_label_conflict_texts", len(train_conflicts)],
    ["unseen_test_label_conflict_texts", len(test_conflicts_unseen)],
    ["strict_development_rows", len(train_strict)],
    ["build_rows_used_for_lexicon", len(train_build)],
    ["validation_rows_reserved_for_thresholds", len(train_validation)],
    ["strict_unseen_test_rows_reserved", len(test_strict)],
], columns=["item", "count"])
summary.to_csv(AUD / "shonasenti_split_audit_summary.csv", index=False)

train_unique.to_csv(AUD / "train_unique.csv", index=False)
test_unseen_unique.to_csv(AUD / "test_unseen_unique.csv", index=False)
train_strict.to_csv(AUD / "train_strict_development_pool.csv", index=False)
train_build.to_csv(AUD / "train_build_for_lexicon.csv", index=False)
train_validation.to_csv(AUD / "train_validation_for_thresholds.csv", index=False)
test_strict.to_csv(AUD / "test_strict_unseen.csv", index=False)
train_raw[train_raw["_norm_text"].isin(overlap_exact)].to_csv(AUD / "train_rows_with_test_overlap.csv", index=False)
test_raw[test_raw["_norm_text"].isin(overlap_exact)].to_csv(AUD / "test_rows_with_train_overlap.csv", index=False)

conflict_rows = pd.concat([
    train_raw[train_raw["_norm_text"].isin(train_conflicts)].assign(conflict_source="train"),
    test_raw[test_raw["_norm_text"].isin(test_conflicts_all)].assign(conflict_source="test")
], ignore_index=True)
conflict_rows.to_csv(AUD / "label_conflict_rows.csv", index=False)

pd.DataFrame({
    "class": ["NEG", "NEU", "POS"],
    "build": train_build["_label"].value_counts().reindex(["NEG", "NEU", "POS"], fill_value=0).to_numpy(),
    "validation": train_validation["_label"].value_counts().reindex(["NEG", "NEU", "POS"], fill_value=0).to_numpy(),
    "strict_test": test_strict["_label"].value_counts().reindex(["NEG", "NEU", "POS"], fill_value=0).to_numpy(),
}).to_csv(AUD / "strict_split_class_distribution.csv", index=False)

write_json(AUD / "clean_data_boundary.json", {
    "strict_development_rows": int(len(train_strict)),
    "build_rows_used_for_candidate_discovery": int(len(train_build)),
    "validation_rows_reserved_for_thresholds": int(len(train_validation)),
    "strict_unseen_test_rows": int(len(test_strict)),
    "test_used_for_candidate_selection": False,
    "validation_used_for_candidate_selection": False,
    "test_used_for_threshold_selection": False,
    "overlap_rule": "Exact NFKC/casefold/whitespace-normalised duplicates are kept in train and removed from test.",
    "conflict_rule": "Texts with conflicting gold labels are excluded from the strict run and retained in the audit.",
    "lexical_overlap_is_sensitivity_only": True,
})

print(summary.to_string(index=False))
print("\nBuild-split labels used for PMI:")
print(train_build["_label"].value_counts().to_string())
print("\nValidation rows reserved for thresholds:", len(train_validation))
print("\nThe strict test is reserved; its labels are not used in construction or threshold selection.")

# Partition integrity is verified against the frozen source boundary.

# Source data audit figures
schema = pd.DataFrame({"Column": sorted(set(train_raw.columns) | set(test_raw.columns))})
schema["Train"] = schema["Column"].isin(train_raw.columns).map({True: "Present", False: "Missing"})
schema["Test"] = schema["Column"].isin(test_raw.columns).map({True: "Present", False: "Missing"})
save_table_image(schema, "Schema comparison of ShonaSenti training and test subsets", "fig_5_1_schema_compatibility.png", max_rows=35, fontsize=7)
copy_alias(FIG / "fig_5_1_schema_compatibility.png", "shonasenti_schema_table_pretty_v2.png")

split_counts = pd.DataFrame({
    "Split": ["Published train", "Published test", "Unique train", "Build", "Validation", "Strict unseen test"],
    "Rows": [len(train_raw), len(test_raw), len(train_unique), len(train_build), len(train_validation), len(test_strict)]
})
plt.figure(figsize=(9, 4.8))
bars = plt.bar(split_counts["Split"], split_counts["Rows"])
plt.ylabel("Tweets")
plt.title("ShonaSenti rows retained after duplicate and overlap controls")
plt.xticks(rotation=18, ha="right")
for bar, value in zip(bars, split_counts["Rows"]):
    plt.text(bar.get_x() + bar.get_width()/2, value, f"{value:,}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
save_current_figure(["fig_5_1_dataset_distribution.png", "fig_shonasenti_duplicate_audit.png"])


In [ ]:

# Load and verify the working lexicon before any candidate is removed as already represented

master_lex_input = read_any(MASTER_LEX_PATH)
master_lex = master_lex_input.copy()
shona_col = find_col_ci(master_lex, ["Shona"])
expanded_shona_col = find_col_ci(master_lex, ["expanded_shona"])
score_col = find_col_ci(master_lex, ["Score"])
english_col = find_col_ci(master_lex, ["English"])
expanded_class_col_existing = find_col_ci(master_lex, ["expanded_shona_class"])

missing = [name for name, col in [("Shona", shona_col), ("expanded_shona", expanded_shona_col), ("Score", score_col)] if col is None]
if missing:
    raise ValueError(f"Base lexicon is missing required columns: {missing}")

master_lex = master_lex.copy()
master_lex["_shona_norm"] = master_lex[shona_col].astype("string").map(normalize_text)
master_lex["_expanded_norm"] = master_lex[expanded_shona_col].astype("string").map(normalize_text)

lex_forms = set(master_lex["_shona_norm"].dropna()) | set(master_lex["_expanded_norm"].dropna())
lex_forms.discard("")
lex_english = set()
if english_col:
    lex_english = set(master_lex[english_col].astype("string").map(normalize_text).dropna())
    lex_english.discard("")

marker_issues = []
for column in ["v2_validation_status", "v2_sentiment_basis", "v2_training_support"]:
    if column in master_lex.columns and master_lex[column].notna().any():
        marker_issues.append(f"non-empty {column}")
if expanded_class_col_existing:
    values = master_lex[expanded_class_col_existing].astype("string").fillna("").str.casefold()
    if values.str.contains(r"pmi|train_only|native_consensus|lexicon_addition", regex=True).any():
        marker_issues.append("expanded_shona_class contains prior addition markers")

base_check = {
    "expected_rows": EXPECTED_INPUT_BASE_LEXICON_ROWS,
    "observed_rows": int(len(master_lex)),
    "matches_expected": bool(len(master_lex) == EXPECTED_INPUT_BASE_LEXICON_ROWS),
    "marker_issues": marker_issues,
    "source_file": str(MASTER_LEX_PATH),
    "source_sha256": sha256_file(MASTER_LEX_PATH),
}
write_json(AUD / "base_lexicon_check.json", base_check)

if ENFORCE_INPUT_BASE_LEXICON_ROWS and len(master_lex) != EXPECTED_INPUT_BASE_LEXICON_ROWS:
    raise ValueError(f"Base lexicon has {len(master_lex)} rows; expected {EXPECTED_INPUT_BASE_LEXICON_ROWS}.")
if ENFORCE_CLEAN_BASE_MARKERS and marker_issues:
    raise ValueError("The selected base lexicon appears to contain prior additions: " + "; ".join(marker_issues))

# Audit inherited score integrity before the base is used for candidate matching or evaluation.
# The archival 6,632-row resource is preserved. Rows failing the fixed score criteria are
# quarantined and excluded from the evaluation-ready base; they are not silently deleted.
base_score_text = master_lex_input[score_col].astype("string").fillna("").str.strip()
base_numeric_scores = pd.to_numeric(master_lex_input[score_col], errors="coerce")
base_missing_scores = base_score_text.eq("")
base_invalid_nonblank = base_score_text.ne("") & base_numeric_scores.isna()
base_below_range = base_numeric_scores.notna() & (base_numeric_scores < BASE_SCORE_MIN)
base_above_range = base_numeric_scores.notna() & (base_numeric_scores > BASE_SCORE_MAX)
base_quarantine_mask = (
    base_missing_scores | base_invalid_nonblank | base_below_range | base_above_range
)

def inherited_score_quarantine_reason(row_index):
    reasons = []
    if bool(base_missing_scores.loc[row_index]):
        reasons.append("missing_score")
    if bool(base_invalid_nonblank.loc[row_index]):
        reasons.append("non_numeric_score")
    if bool(base_below_range.loc[row_index]):
        reasons.append("score_below_minus_9")
    if bool(base_above_range.loc[row_index]):
        reasons.append("score_above_9")
    return ";".join(reasons)

base_quarantine = master_lex_input.loc[base_quarantine_mask].copy()
base_quarantine.insert(
    0, "source_data_row_number_1_based", base_quarantine.index + 1
)
base_quarantine.insert(
    1, "source_spreadsheet_row_number_1_based", base_quarantine.index + 2
)
base_quarantine["quarantine_reason"] = [
    inherited_score_quarantine_reason(i) for i in base_quarantine.index
]
base_quarantine["quarantine_decision"] = "excluded_from_evaluation_ready_base"
base_quarantine["resource_lineage_status"] = "inherited_score_integrity_issue"

# Canonical signature makes sure a different set of 17 rows is not silently quarantined.
signature_records = []
for source_index, row in master_lex_input.loc[base_quarantine_mask].iterrows():
    record = {"data_row_number_1_based": int(source_index + 1)}
    for column in master_lex_input.columns:
        value = row[column]
        record[column] = "" if pd.isna(value) else str(value).strip()
    signature_records.append(record)
quarantine_payload = json.dumps(
    signature_records, ensure_ascii=False, sort_keys=True, separators=(",", ":")
).encode("utf-8")
observed_quarantine_signature = hashlib.sha256(quarantine_payload).hexdigest()

base_quarantine.to_csv(
    AUD / "base_lexicon_quarantined_score_rows.csv", index=False
)

# The evaluation-ready base is the source resource minus the quarantined rows.
master_lex = master_lex_input.loc[~base_quarantine_mask].reset_index(drop=True).copy()
clean_base_export = master_lex.copy()
clean_base_path = ROOT_OUT / "FrenchyShona_base_6615_evaluation_ready.csv"
clean_base_export.to_csv(clean_base_path, index=False)

if ENFORCE_QUARANTINE_COUNT and len(base_quarantine) != EXPECTED_QUARANTINED_SCORE_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_QUARANTINED_SCORE_ROWS} inherited score-integrity rows, "
        f"but found {len(base_quarantine)}. Inspect the quarantine audit before continuing."
    )
if ENFORCE_QUARANTINE_SIGNATURE and observed_quarantine_signature != EXPECTED_QUARANTINE_SIGNATURE_SHA256:
    raise ValueError(
        "The rows failing the score-integrity criteria do not match the audited set. "
        "Do not continue until the source lexicon is reconciled."
    )
if ENFORCE_CLEAN_BASE_ROWS and len(master_lex) != EXPECTED_EVALUATION_BASE_LEXICON_ROWS:
    raise ValueError(
        f"Evaluation-ready base has {len(master_lex)} rows; "
        f"expected {EXPECTED_EVALUATION_BASE_LEXICON_ROWS}."
    )

# Validate the cleaned base and record a canonical core hash.
clean_numeric_scores = pd.to_numeric(master_lex[score_col], errors="coerce")
if clean_numeric_scores.isna().any():
    raise ValueError("The evaluation-ready base still contains a missing or nonnumeric score.")
if ((clean_numeric_scores < BASE_SCORE_MIN) | (clean_numeric_scores > BASE_SCORE_MAX)).any():
    raise ValueError("The evaluation-ready base still contains a score outside [-9, 9].")

clean_core = master_lex[[shona_col, expanded_shona_col, score_col]].copy()
clean_core.columns = ["Shona", "expanded_shona", "Score"]
clean_core["Shona"] = clean_core["Shona"].astype("string").fillna("").map(normalize_text)
clean_core["expanded_shona"] = (
    clean_core["expanded_shona"].astype("string").fillna("").map(normalize_text)
)
clean_core["Score"] = [
    format(float(value), ".12g") for value in clean_numeric_scores
]
clean_core = clean_core.sort_values(
    ["Shona", "expanded_shona", "Score"], kind="mergesort"
).reset_index(drop=True)
clean_core_payload = clean_core.to_csv(
    index=False, lineterminator="\n"
).encode("utf-8")
observed_clean_core_sha256 = hashlib.sha256(clean_core_payload).hexdigest()
if observed_clean_core_sha256 != EXPECTED_CLEAN_BASE_CORE_SHA256:
    raise ValueError(
        "The 6,615-row clean-base core hash differs from the audited resource. "
        "Inspect the input and quarantine files before continuing."
    )

base_score_audit = {
    "source_rows": int(len(master_lex_input)),
    "missing_score_count": int(base_missing_scores.sum()),
    "non_numeric_nonblank_count": int(base_invalid_nonblank.sum()),
    "below_range_count": int(base_below_range.sum()),
    "above_range_count": int(base_above_range.sum()),
    "quarantined_rows": int(len(base_quarantine)),
    "evaluation_ready_rows": int(len(master_lex)),
    "score_min_after_quarantine": float(clean_numeric_scores.min()),
    "score_max_after_quarantine": float(clean_numeric_scores.max()),
    "quarantine_signature_sha256": observed_quarantine_signature,
    "evaluation_ready_core_sha256": observed_clean_core_sha256,
    "criterion": "score must be numeric, non-missing and within [-9, 9]",
    "quarantine_status": "retained in audit; excluded from evaluation-ready base",
}
write_json(AUD / "base_lexicon_score_quarantine_summary.json", base_score_audit)

# Recompute normalised forms after quarantine. Quarantined-only forms are blocked from
# re-entering the same run as newly discovered additions.
master_lex["_shona_norm"] = master_lex[shona_col].astype("string").map(normalize_text)
master_lex["_expanded_norm"] = (
    master_lex[expanded_shona_col].astype("string").map(normalize_text)
)
lex_forms = set(master_lex["_shona_norm"].dropna()) | set(
    master_lex["_expanded_norm"].dropna()
)
lex_forms.discard("")

quarantine_shona_norm = (
    base_quarantine[shona_col].astype("string").fillna("").map(normalize_text)
)
quarantine_expanded_norm = (
    base_quarantine[expanded_shona_col].astype("string").fillna("").map(normalize_text)
)
quarantined_lex_forms_all = set(quarantine_shona_norm) | set(quarantine_expanded_norm)
quarantined_lex_forms_all.discard("")
quarantined_only_forms = quarantined_lex_forms_all - lex_forms

write_json(AUD / "base_quarantined_form_overlap_with_clean_base.json", {
    "all_nonblank_quarantined_forms": len(quarantined_lex_forms_all),
    "quarantined_forms_still_represented_by_clean_rows": len(
        quarantined_lex_forms_all & lex_forms
    ),
    "quarantined_only_forms_blocked_from_rederivation": len(quarantined_only_forms),
})

base_form_rows = pd.concat([
    master_lex[[shona_col, score_col]].rename(columns={shona_col: "form", score_col: "score"}),
    master_lex[[expanded_shona_col, score_col]].rename(columns={expanded_shona_col: "form", score_col: "score"}),
], ignore_index=True)
base_form_rows["form_norm"] = base_form_rows["form"].astype("string").fillna("").map(normalize_text)
base_form_rows = base_form_rows[base_form_rows["form_norm"].ne("")].copy()
base_form_conflicts = (
    base_form_rows.groupby("form_norm")["score"]
    .agg(
        n_rows="size",
        n_scores=lambda s: pd.to_numeric(s, errors="coerce").nunique(),
        scores=lambda s: "|".join(map(str, sorted(set(pd.to_numeric(s, errors="coerce").dropna())))),
    )
    .reset_index()
)
base_form_conflicts = base_form_conflicts[base_form_conflicts["n_scores"] > 1]
base_form_conflicts.to_csv(AUD / "base_lexicon_form_score_conflicts.csv", index=False)


def write_lexicon_token_matchability_audit(frame, form_columns, score_column, prefix):
    """Describe which lexicon forms can enter the exact token matcher used in evaluation."""
    rows = []
    for row_position, (_, row) in enumerate(frame.iterrows(), start=1):
        score_value = row.get(score_column, pd.NA) if score_column else pd.NA
        for source_column in form_columns:
            raw_value = row.get(source_column, "")
            raw_form = "" if pd.isna(raw_value) else str(raw_value)
            normalised_form = normalize_text(raw_form)
            cleaned_form = clean_lexical_text(raw_form)
            tweet_tokens = tokenize_lexical_text(raw_form)
            score_map_key = norm_token(normalised_form) if normalised_form else ""

            if not normalised_form:
                status = "blank_form"
            elif " " in normalised_form:
                status = "multiword_excluded_token_only"
            elif len(tweet_tokens) == 0:
                status = "empty_after_tweet_tokenisation"
            elif len(tweet_tokens) > 1:
                status = "split_into_multiple_tweet_tokens"
            elif tweet_tokens[0] != score_map_key:
                status = "normalisation_mismatch_unmatchable"
            else:
                status = "direct_single_token_matchable"

            rows.append({
                "lexicon_row_position": row_position,
                "source_column": source_column,
                "raw_form": raw_form,
                "normalised_form": normalised_form,
                "cleaned_form": cleaned_form,
                "tweet_token_count": len(tweet_tokens),
                "tweet_tokens": "|".join(tweet_tokens),
                "score_map_key": score_map_key,
                "matchability_status": status,
                "score": score_value,
            })

    audit = pd.DataFrame(rows, columns=[
        "lexicon_row_position", "source_column", "raw_form", "normalised_form",
        "cleaned_form", "tweet_token_count", "tweet_tokens", "score_map_key",
        "matchability_status", "score"
    ])
    audit_path = AUD / f"{prefix}_token_matchability_audit.csv"
    audit.to_csv(audit_path, index=False)

    status_counts = audit["matchability_status"].value_counts().to_dict()
    direct = audit[audit["matchability_status"].eq("direct_single_token_matchable")].copy()
    nonblank = audit[audit["matchability_status"].ne("blank_form")]
    duplicate_direct_counts = direct["score_map_key"].value_counts()

    direct_score_conflict_count = 0
    if len(direct):
        direct_scores = direct.copy()
        direct_scores["score_numeric"] = pd.to_numeric(direct_scores["score"], errors="coerce")
        direct_score_conflict_count = int(
            (direct_scores.groupby("score_map_key")["score_numeric"].nunique(dropna=True) > 1).sum()
        )

    summary = {
        "prefix": prefix,
        "lexicon_rows": int(len(frame)),
        "form_columns": list(form_columns),
        "form_entries_audited": int(len(audit)),
        "nonblank_form_entries": int(len(nonblank)),
        "direct_single_token_entries": int(len(direct)),
        "direct_single_token_unique_match_keys": int(direct["score_map_key"].nunique()),
        "direct_matchability_percent_of_nonblank_entries": (
            float(100 * len(direct) / len(nonblank)) if len(nonblank) else None
        ),
        "multiword_entries_excluded": int(status_counts.get("multiword_excluded_token_only", 0)),
        "empty_after_tweet_tokenisation": int(status_counts.get("empty_after_tweet_tokenisation", 0)),
        "split_into_multiple_tweet_tokens": int(status_counts.get("split_into_multiple_tweet_tokens", 0)),
        "normalisation_mismatch_unmatchable": int(status_counts.get("normalisation_mismatch_unmatchable", 0)),
        "blank_form_entries": int(status_counts.get("blank_form", 0)),
        "duplicate_direct_match_keys": int((duplicate_direct_counts > 1).sum()),
        "duplicate_direct_match_keys_with_score_conflicts": direct_score_conflict_count,
        "matching_rule": "exact token match after project normalisation",
        "multiword_entries_in_token_only_evaluation": False,
        "status_counts": {str(k): int(v) for k, v in status_counts.items()},
    }
    write_json(AUD / f"{prefix}_token_matchability_summary.json", summary)
    return audit, summary


base_token_matchability_audit, base_token_matchability_summary = (
    write_lexicon_token_matchability_audit(
        master_lex,
        [shona_col, expanded_shona_col],
        score_col,
        "base_lexicon",
    )
)

print("Input working lexicon rows:", len(master_lex_input))
print("Quarantined inherited score-integrity rows:", len(base_quarantine))
print("Evaluation-ready base rows:", len(master_lex))
print("Working lexicon rows:", len(master_lex))
print("Unique Shona/expanded forms:", len(lex_forms))
print("Base lexicon score quarantine and clean-base audit: PASS")
print("Directly matchable unique base forms:", base_token_matchability_summary["direct_single_token_unique_match_keys"])

# Optional provenance and schema figures are generated when supporting inputs are available.
if ORIGINAL_LEX_PATH:
    original_lex = pd.read_excel(ORIGINAL_LEX_PATH)
    expanded_for_schema = read_any(WORKING_EXPANDED_XLSX) if WORKING_EXPANDED_XLSX else master_lex.drop(columns=["_shona_norm", "_expanded_norm"])
    schema_df = pd.DataFrame([
        ["Entries", len(original_lex), len(expanded_for_schema)],
        ["Columns", original_lex.shape[1], expanded_for_schema.shape[1]],
        ["Includes Shona", "Yes" if find_col_ci(original_lex, ["Shona"]) else "No", "Yes" if find_col_ci(expanded_for_schema, ["Shona"]) else "No"],
    ], columns=["Feature", "Original FrenchyLuba", "Working expanded lexicon"])
    save_table_image(schema_df, "Schema and structural comparison of the lexicons", "fig_5_6_lexicon_schema_structure.png")
else:
    mark_figure_pending("fig_5_6_lexicon_schema_structure.png", "Original FrenchyLuba file not supplied")

if A2_GT_PATH:
    gt = pd.read_csv(A2_GT_PATH)
    a_col, b_col = find_col_ci(gt, ["Shona"]), find_col_ci(gt, ["Shona_gt"])
    if a_col and b_col:
        a = gt[a_col].astype("string").map(normalize_text)
        b = gt[b_col].astype("string").map(normalize_text)
        comparable = a.ne("") & b.ne("")
        values = [int((a[comparable] == b[comparable]).sum()), int((a[comparable] != b[comparable]).sum())]
        plt.figure(figsize=(6, 4))
        bars = plt.bar(["Match", "Different"], values)
        plt.ylabel("Rows")
        plt.title("Existing Shona compared with Shona_gt")
        for bar, value in zip(bars, values):
            plt.text(bar.get_x() + bar.get_width()/2, value, f"{value:,}", ha="center", va="bottom")
        plt.tight_layout()
        save_current_figure("fig_shona_vs_shona_gt_summary.png")


In [ ]:

# Tokenise Shona build texts and apply auditable filtering and review flags

train_work = train_build.copy()

def enforce_train_work_boundary(stage):
    actual_texts = frozenset(train_work["_norm_text"])
    actual_digest = frame_text_label_sha256(train_work)
    record = {
        "stage": stage,
        "expected_rows": BOUNDARY_BUILD_ROWS,
        "actual_rows": int(len(train_work)),
        "expected_build_text_label_sha256": BOUNDARY_BUILD_DIGEST,
        "actual_build_text_label_sha256": actual_digest,
        "text_set_matches_frozen_build_split": actual_texts == BOUNDARY_BUILD_TEXTS,
        "validation_overlap": len(actual_texts & BOUNDARY_VALIDATION_TEXTS),
        "strict_test_overlap": len(actual_texts & BOUNDARY_TEST_TEXTS),
        "unique_normalised_texts": bool(train_work["_norm_text"].is_unique),
    }
    record["pass"] = bool(
        record["actual_rows"] == record["expected_rows"]
        and record["actual_build_text_label_sha256"] == record["expected_build_text_label_sha256"]
        and record["text_set_matches_frozen_build_split"]
        and record["validation_overlap"] == 0
        and record["strict_test_overlap"] == 0
        and record["unique_normalised_texts"]
    )
    write_json(AUD / f"train_work_boundary_{stage}.json", record)
    if not record["pass"]:
        raise AssertionError(
            "The candidate-construction frame no longer equals the declared build split: "
            + json_text(record)
        )
    return record

enforce_train_work_boundary("after_creation")

train_work["_clean"] = train_work[TEXT_COL].map(clean_lexical_text)
train_work["_tokens"] = train_work[TEXT_COL].map(tokenize_lexical_text)

token_counts = Counter(word for words in train_work["_tokens"] for word in words)
vocab = pd.DataFrame(token_counts.items(), columns=["token", "count"])
vocab["len"] = vocab["token"].str.len()

english_stop = set(ENGLISH_STOP_WORDS) | set("""
the and is are was were be been being a an in on at of to for from by with about as into like through after
over between out against during without before under around among this that these those i you he she it we they
them his her their our your my me do does did done have has had will would should can could just now then very
also what who when where why how guys guy
""".split())

# Core Shona stop forms used by the executed filtering procedure.
core_shona_stop = set("""
kuti uye kana asi zvino saka ndiye ndiyo ndiwo pamwe kunge izvi izvo ichi icho iyi iro aya here kwete hongu
nekuti ndiri uri ari tiri muri vari pa ku mu ne ya ye va wo
""".split())

# These forms are flagged as potentially grammatical or deictic rather than sentiment-bearing.
# They are review flags only and are not removed automatically.
suspected_non_content_seed = set("""
umu imomo apa ipapo uko ikoko uyu uyo uyuwo uyowo iyeye iyoyo iwoyo ava avo awa awo idzi idzo
hayo iyo zvayo yowo rowo dzowo wowo vano vane vanga vange ano ane anga ange uno une tino tine mano mane
aizve zvakare futi zve nhaiwe iwe imi isu ivo iye ini ndofunga ndinofunga kungoita kungo unotoona aripo aripa
""".split())

identity_terms = set("""
varungu vatema mabhunu vashona vandebele matebele mandevere vazezuru vakaranga vamanyika vandau vakorekore
vapositori majoni magandanga vabvakure
""".split())

# These can be proper names in some contexts and ordinary content words in others. They are never
# removed solely by prefix stripping.
ambiguous_name_or_content = {"moyo", "angel", "triangle"}

zim_places = set("""
harare bulawayo mutare gweru kwekwe masvingo chinhoyi kadoma marondera bindura victoriafalls vicfalls hwange
kariba rusape chipinge beitbridge plumtree redcliff chegutu norton shamva chiredzi chitungwiza guruve shurugwi chirundu
""".split())
public_names = set("""
chamisa mnangagwa mugabe tsvangirai chiwenga makoni biti kasukuwere chombo mutasa sibanda ndlovu mwonzora
khupe mutsvangwa chinotimba winkyd winky jah jahprayzah jahprayza freeman killer tocky tockyvibes suluman
sulumanchimbetu chimbetu poptain holyten exq shashl nuttyo stunner king98 kikkybadass makandiwa magaya
uzumba mutukudzi mtukudzi prayzah
""".split())
political_terms = {"zanu", "zanupf", "zanu-pf", "vezanu", "mdc", "mdct", "ccc", "pf"}
slang_or_foreign = {"mjolo", "tonaz", "vasatan"}

blocked_exact = (zim_places | public_names | political_terms | slang_or_foreign) - ambiguous_name_or_content
blocked_prefixes = ("mu", "pa", "ku", "va", "wa", "we", "kwa")


def is_blocked_name(token):
    if token in ambiguous_name_or_content:
        return False
    if token in blocked_exact:
        return True
    for prefix in blocked_prefixes:
        if token.startswith(prefix) and len(token) > len(prefix):
            base = token[len(prefix):]
            if base in ambiguous_name_or_content:
                return False
            if base in blocked_exact:
                return True
    return False


def token_noise_reason(token):
    if not token or len(token) < 3:
        return "short"
    if len(token) >= 25:
        return "long"
    if re.search(r"(.)\1\1\1", token):
        return "repeat_noise"
    if WORD_FREQ_AVAILABLE and zipf_frequency(token, "en") > ENGLISH_ZIPF_THRESHOLD:
        return "english_frequency"
    if token in english_stop:
        return "english_stopword"
    if token in core_shona_stop:
        return "core_shona_function_word"
    if is_blocked_name(token):
        return "name_or_location"
    return ""


def review_flags_for(token, noise_reason):
    if noise_reason:
        return ""
    flags = []
    if token in suspected_non_content_seed:
        flags.append("suspected_non_content_seed")
    if token in identity_terms:
        flags.append("identity_term")
    if token in ambiguous_name_or_content:
        flags.append("ambiguous_name_or_content")
    for prefix in blocked_prefixes:
        if token.startswith(prefix) and token[len(prefix):] in ambiguous_name_or_content:
            flags.append("ambiguous_name_or_content")
    return ";".join(sorted(set(flags)))


vocab["noise_reason"] = vocab["token"].map(token_noise_reason)
vocab["label"] = np.where(
    vocab["noise_reason"].ne(""), "noise_rule",
    np.where(vocab["count"].le(2), "rare_review", "shona_candidate")
)
vocab["review_flag"] = [review_flags_for(t, r) for t, r in zip(vocab["token"], vocab["noise_reason"])]
REVIEW_FLAGS = dict(zip(vocab["token"], vocab["review_flag"]))

_build_document_support = Counter()
for _tokens in train_work["_tokens"]:
    for _token in set(_tokens):
        _build_document_support[_token] += 1
vocab["document_support"] = vocab["token"].map(_build_document_support).fillna(0).astype(int)

def safe_json_object(text):
    text = str(text or "").strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)


def normalise_confidence(value):
    if isinstance(value, str):
        label = normalize_text(value)
        mapped = {"low": 0.35, "medium": 0.65, "high": 0.90}
        if label in mapped:
            return mapped[label]
    try:
        return float(np.clip(float(value), 0.0, 1.0))
    except Exception:
        return 0.0


def load_rag_client():
    if not USE_RAG_GPT:
        return None
    try:
        from kaggle_secrets import UserSecretsClient
        from openai import OpenAI
        api_key = UserSecretsClient().get_secret(OPENAI_SECRET_NAME)
        if not api_key:
            raise RuntimeError("empty secret")
        return OpenAI(api_key=api_key)
    except Exception as exc:
        raise RuntimeError(
            f"RAG/GPT is enabled. Add the Kaggle secret {OPENAI_SECRET_NAME!r} "
            "and allow this notebook to use it."
        ) from exc


RAG_CLIENT = load_rag_client()
RAG_CACHE_DIR = Path(RAG_CACHE_DIRECTORY).expanduser()
try:
    RAG_CACHE_DIR.mkdir(parents=True, exist_ok=True)
except Exception:
    RAG_CACHE_DIR = ROOT_OUT.parent / "FrenchyShona_V2_RAG_Cache_v7_2"
    RAG_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def sample_build_contexts(token, max_contexts=6, include_labels=False):
    mask = train_work["_tokens"].apply(lambda words: token in set(words))
    hits = train_work.loc[mask, [TEXT_COL, "_label", "_clean"]].copy()
    if hits.empty:
        return []
    chosen_indices = []
    for label in ["NEG", "NEU", "POS"]:
        part = hits[hits["_label"].eq(label)]
        if len(part):
            chosen_indices.append(
                part.sample(1, random_state=SEED + len(chosen_indices)).index[0]
            )
    if len(chosen_indices) < max_contexts:
        remaining = hits.index.difference(chosen_indices)
        extra_n = min(max_contexts - len(chosen_indices), len(remaining))
        if extra_n:
            chosen_indices.extend(
                hits.loc[remaining].sample(extra_n, random_state=SEED).index.tolist()
            )
    output = []
    for idx in chosen_indices[:max_contexts]:
        row = hits.loc[idx]
        item = {"text": str(row["_clean"])}
        if include_labels:
            item["label"] = str(row["_label"])
        output.append(item)
    return output


def load_jsonl_cache(path):
    cache = {}
    if not path.exists():
        return cache
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if obj.get("cache_key"):
                    cache[obj["cache_key"]] = obj
            except Exception:
                continue
    return cache


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def rag_cache_key(task_name, prompt_version, payload):
    canonical = {
        "task": task_name,
        "prompt_version": prompt_version,
        "model": RAG_MODEL,
        "payload": payload,
    }
    raw = json.dumps(canonical, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def call_rag_json(task_name, prompt_version, system_text, payload):
    cache_path = RAG_CACHE_DIR / f"{task_name}.jsonl"
    cache = load_jsonl_cache(cache_path)
    key = rag_cache_key(task_name, prompt_version, payload)
    if key in cache:
        return cache[key]["parsed_response"], True

    user_text = json.dumps(payload, ensure_ascii=False)
    last_error = None
    for attempt in range(1, RAG_MAX_RETRIES + 1):
        try:
            kwargs = {
                "model": RAG_MODEL,
                "messages": [
                    {"role": "system", "content": system_text},
                    {"role": "user", "content": user_text},
                ],
                "response_format": {"type": "json_object"},
            }
            try:
                response = RAG_CLIENT.chat.completions.create(**kwargs)
            except Exception as first_exc:
                # Some SDK/model combinations do not expose response_format. Retry without it.
                if "response_format" not in str(first_exc).lower():
                    raise
                kwargs.pop("response_format", None)
                response = RAG_CLIENT.chat.completions.create(**kwargs)

            raw_text = response.choices[0].message.content
            parsed = safe_json_object(raw_text)
            record = {
                "cache_key": key,
                "task": task_name,
                "prompt_version": prompt_version,
                "model": RAG_MODEL,
                "response_id": getattr(response, "id", None),
                "parsed_response": parsed,
            }
            append_jsonl(cache_path, record)
            return parsed, False
        except Exception as exc:
            last_error = exc
            if attempt < RAG_MAX_RETRIES:
                time.sleep(RAG_SLEEP_SECONDS * attempt)
    raise RuntimeError(
        f"Contextual model request failed after {RAG_MAX_RETRIES} attempts: {last_error}"
    )


def batch_items(items, size):
    for start_index in range(0, len(items), size):
        yield items[start_index:start_index + size]


def schema_sha256(schema):
    raw = json.dumps(schema, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def call_rag_json_schema(task_name, prompt_version, system_text, payload, schema_name, schema):
    cache_path = RAG_CACHE_DIR / f"{task_name}.jsonl"
    cache = load_jsonl_cache(cache_path)
    payload_with_schema = {"payload": payload, "schema_sha256": schema_sha256(schema)}
    key = rag_cache_key(task_name, prompt_version, payload_with_schema)
    if key in cache:
        return cache[key]["parsed_response"], True

    user_text = json.dumps(payload, ensure_ascii=False)
    last_error = None
    for attempt in range(1, RAG_MAX_RETRIES + 1):
        try:
            kwargs = {
                "model": RAG_MODEL,
                "messages": [
                    {"role": "system", "content": system_text},
                    {"role": "user", "content": user_text},
                ],
                "response_format": {
                    "type": "json_schema",
                    "json_schema": {"name": schema_name, "strict": True, "schema": schema},
                },
            }
            try:
                response = RAG_CLIENT.chat.completions.create(**kwargs)
            except Exception as schema_exc:
                message = str(schema_exc).lower()
                if not any(term in message for term in ("response_format", "json_schema", "schema")):
                    raise
                kwargs["response_format"] = {"type": "json_object"}
                response = RAG_CLIENT.chat.completions.create(**kwargs)
            parsed = safe_json_object(response.choices[0].message.content)
            append_jsonl(cache_path, {
                "cache_key": key,
                "task": task_name,
                "prompt_version": prompt_version,
                "model": RAG_MODEL,
                "response_id": getattr(response, "id", None),
                "parsed_response": parsed,
            })
            return parsed, False
        except Exception as exc:
            last_error = exc
            if attempt < RAG_MAX_RETRIES:
                time.sleep(RAG_SLEEP_SECONDS * attempt)
    raise RuntimeError(
        f"Structured contextual request failed after {RAG_MAX_RETRIES} attempts: {last_error}"
    )


RAG_WORD_TYPES = {
    "noun", "verb", "adjective", "adverb", "interjection", "idiom", "phrase",
    "pronoun", "deictic", "demonstrative", "function", "particle", "conjunction",
    "preposition", "concord", "auxiliary", "name", "place", "identity", "unknown",
}
RAG_CONTENT_WORD_TYPES = {"noun", "verb", "adjective", "adverb", "interjection", "idiom", "phrase"}

RAG_CONTEXT_ITEM_SCHEMA = {
    "type": "object",
    "properties": {
        "candidate_id": {"type": "string", "minLength": 1},
        "word": {"type": "string", "minLength": 1},
        "valid_shona_proposal": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "english_meaning": {"type": "string", "minLength": 1},
        "word_type": {"type": "string", "enum": sorted(RAG_WORD_TYPES)},
        "lexical_sentiment": {"type": "string", "enum": ["NEG", "NEU", "POS", "Context-dependent", "Not-applicable"]},
        "corpus_association": {"type": "string", "enum": ["NEG", "NEU", "POS", "No-stable-association"]},
        "sentiment_basis_proposal": {"type": "string", "enum": ["lexical_polarity", "corpus_association", "context_dependent", "not_applicable"]},
        "provisional_score_minus9_to9": {"type": "integer", "minimum": -9, "maximum": 9},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "lexical_status": {"type": "string", "enum": ["support", "review", "reject"]},
        "reason": {"type": "string", "minLength": 1},
    },
    "required": [
        "candidate_id", "word", "valid_shona_proposal", "english_meaning", "word_type",
        "lexical_sentiment", "corpus_association", "sentiment_basis_proposal",
        "provisional_score_minus9_to9", "confidence", "lexical_status", "reason"
    ],
    "additionalProperties": False,
}
RAG_CONTEXT_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {"items": {"type": "array", "items": RAG_CONTEXT_ITEM_SCHEMA}},
    "required": ["items"],
    "additionalProperties": False,
}


def validate_context_item(obj, expected_id, expected_word):
    problems = []
    if not isinstance(obj, dict):
        return ["not_an_object"]
    candidate_id = str(obj.get("candidate_id", "") or "").strip().upper()
    word = normalize_text(obj.get("word", ""))
    if candidate_id != expected_id:
        problems.append("candidate_id_mismatch")
    if word != expected_word:
        problems.append("word_mismatch")
    if obj.get("valid_shona_proposal") not in {"yes", "no", "uncertain"}:
        problems.append("invalid_valid_shona_proposal")
    if not str(obj.get("english_meaning", "") or "").strip():
        problems.append("missing_english_meaning")
    if obj.get("word_type") not in RAG_WORD_TYPES:
        problems.append("invalid_word_type")
    if obj.get("lexical_sentiment") not in {"NEG", "NEU", "POS", "Context-dependent", "Not-applicable"}:
        problems.append("invalid_lexical_sentiment")
    association = obj.get("corpus_association")
    if association not in {"NEG", "NEU", "POS", "No-stable-association"}:
        problems.append("invalid_corpus_association")
    if obj.get("sentiment_basis_proposal") not in {"lexical_polarity", "corpus_association", "context_dependent", "not_applicable"}:
        problems.append("invalid_sentiment_basis")
    try:
        score = int(obj.get("provisional_score_minus9_to9"))
        if not -9 <= score <= 9:
            problems.append("score_out_of_range")
    except Exception:
        score = None
        problems.append("invalid_score")
    try:
        confidence = float(obj.get("confidence"))
        if not 0 <= confidence <= 1:
            problems.append("confidence_out_of_range")
    except Exception:
        problems.append("invalid_confidence")
    if obj.get("lexical_status") not in {"support", "review", "reject"}:
        problems.append("invalid_lexical_status")
    if not str(obj.get("reason", "") or "").strip():
        problems.append("missing_reason")
    if score is not None:
        if association == "POS" and score <= 0:
            problems.append("positive_association_requires_positive_score")
        elif association == "NEG" and score >= 0:
            problems.append("negative_association_requires_negative_score")
        elif association in {"NEU", "No-stable-association"} and score != 0:
            problems.append("nonpolar_association_requires_zero_score")
    if obj.get("valid_shona_proposal") == "no" and obj.get("lexical_status") == "support":
        problems.append("invalid_shona_cannot_be_support")
    return problems

vocab.sort_values(["label", "count"], ascending=[True, False]).to_csv(TAB / "vocab_labeled_v2.csv", index=False)
flag_audit = vocab[vocab["review_flag"].ne("") | vocab["noise_reason"].ne("")].copy()
flag_audit.to_csv(TAB / "candidate_filter_and_flag_audit.csv", index=False)

# A separate form records researcher assessment of the filtering seed lists.
# These responses do not alter the run automatically; final word-type decisions are recorded
# during the structured researcher review.
seed_review = pd.DataFrame(
    [{"token": t, "seed_type": "suspected_non_content", "verified_by": "", "decision": "", "comment": ""}
     for t in sorted(suspected_non_content_seed)]
    + [{"token": t, "seed_type": "identity_term", "verified_by": "", "decision": "", "comment": ""}
       for t in sorted(identity_terms)]
    + [{"token": t, "seed_type": "ambiguous_name_or_content", "verified_by": "", "decision": "", "comment": ""}
       for t in sorted(ambiguous_name_or_content)]
)
seed_review.to_csv(
    REV / "single_researcher_filter_seed_review_template.csv", index=False
)

print("Build tweets used for candidate discovery:", len(train_work))
print("Token occurrences:", int(vocab["count"].sum()))
print("Unique tokens:", len(vocab))
print(vocab["label"].value_counts().to_string())
print("Flagged suspected non-content forms:", int(vocab["review_flag"].str.contains("suspected_non_content_seed", na=False).sum()))
print("Flagged identity terms:", int(vocab["review_flag"].str.contains("identity_term", na=False).sum()))

# Build-corpus lexical profile figures
_top20 = vocab.sort_values("count", ascending=False).head(20).sort_values("count")
plt.figure(figsize=(8, 6))
plt.barh(_top20["token"], _top20["count"])
plt.xlabel("Token count")
plt.title("Top 20 tokens in the FrenchyShona V2 build split")
plt.tight_layout()
save_current_figure(["fig_5_2_top20_tokens.png", "top20_tokens_cleaned_shonasenti.png"])

try:
    from wordcloud import WordCloud
    cloud_vocab = vocab[vocab["label"].eq("shona_candidate")]
    frequencies = dict(zip(cloud_vocab["token"], cloud_vocab["count"]))
    if frequencies:
        cloud = WordCloud(width=1600, height=900, background_color="white", random_state=SEED).generate_from_frequencies(frequencies)
        plt.figure(figsize=(12, 7))
        plt.imshow(cloud, interpolation="bilinear")
        plt.axis("off")
        plt.title("Filtered Shona vocabulary after noise reduction")
        save_current_figure(["shona_filtered_wordcloud_clean.png", "fig_5_3_filtered_shona_vocabulary.png"])
except Exception as exc:
    mark_figure_pending("shona_filtered_wordcloud_clean.png", f"wordcloud unavailable: {exc}")

freqs = np.sort(vocab.loc[vocab["noise_reason"].eq(""), "count"].to_numpy())[::-1]
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(freqs) + 1), freqs)
plt.xscale("log"); plt.yscale("log")
plt.xlabel("Token rank"); plt.ylabel("Frequency")
plt.title("Long-tail distribution of the cleaned build-split vocabulary")
plt.tight_layout()
save_current_figure(["long_tail_final.png", "fig_5_4_long_tail.png"])

order = ["noise_rule", "rare_review", "shona_candidate"]
counts = vocab["label"].value_counts().reindex(order).fillna(0)
plt.figure(figsize=(8, 4.5))
bars = plt.barh(order, counts.values)
plt.xlabel("Unique tokens")
plt.title("Token classification after filtering")
for bar, value in zip(bars, counts.values):
    plt.text(value, bar.get_y() + bar.get_height()/2, f" {int(value):,}", va="center")
plt.tight_layout()
save_current_figure(["token_classification_overview_final.png", "fig_5_5_token_classification.png"])


In [ ]:
# Candidate discovery: deterministic filtering plus contextual recovery

enforce_train_work_boundary("before_rag_candidate_recovery")

deterministic_candidates = vocab[vocab["label"].eq("shona_candidate")].copy()
deterministic_candidates["candidate_source"] = "deterministic"

# Revisit only build-vocabulary forms rejected by the English-frequency rule and capable
# of passing the main support floor. Forms already represented in the clean lexicon are
# excluded before API screening.
rag_recovery_pool = vocab[
    vocab["noise_reason"].isin(RAG_RECOVERY_NOISE_REASONS)
    & (vocab["document_support"] >= RAG_RECOVERY_MIN_DOCUMENT_SUPPORT)
    & ~vocab["token"].isin(lex_forms)
    & ~vocab["token"].isin(quarantined_lex_forms_all)
].copy().sort_values(
    ["document_support", "token"], ascending=[False, True]
).reset_index(drop=True)

recovery_input_items = []
for _, row in rag_recovery_pool.iterrows():
    recovery_input_items.append({
        "token": row["token"],
        "document_support": int(row["document_support"]),
        "contexts": sample_build_contexts(
            row["token"], max_contexts=3, include_labels=False
        ),
    })

recovery_results = {}
recovery_cache_hits = 0
recovery_api_calls = 0
recovery_system = (
    "Classify lexical tokens from Shona social-media text using only the supplied "
    "build-corpus contexts. Return one result for every input token in a JSON object "
    "with an items array. Do not assign sentiment."
)

for batch in batch_items(recovery_input_items, RAG_RECOVERY_BATCH_SIZE):
    payload = {
        "task": "language_and_word_type_screen",
        "allowed_lexical_classes": [
            "shona_content", "shona_noncontent", "english_or_code_switch",
            "name_place_or_brand", "uncertain"
        ],
        "allowed_word_types": [
            "noun", "verb", "adjective", "adverb", "interjection", "idiom",
            "pronoun", "deictic", "demonstrative", "function", "particle",
            "conjunction", "preposition", "concord", "auxiliary", "name",
            "place", "identity", "unknown"
        ],
        "rules": [
            "A Shona inflected form may resemble an English word.",
            "Classify the token itself, not the sentiment of the surrounding tweet.",
            "Use shona_content only for a content-bearing Shona lexical form.",
            "Return confidence as a number from 0 to 1."
        ],
        "items": batch,
    }
    response_obj, cache_hit = call_rag_json(
        "candidate_recovery",
        RAG_RECOVERY_PROMPT_VERSION,
        recovery_system,
        payload,
    )
    recovery_cache_hits += int(cache_hit)
    recovery_api_calls += int(not cache_hit)
    for result in response_obj.get("items", []):
        token = normalize_text(result.get("token", "")).replace(" ", "")
        if token:
            recovery_results[token] = result

# Retry any missing items individually; an incomplete batch is not silently accepted.
for item in recovery_input_items:
    token = item["token"]
    if token in recovery_results:
        continue
    payload = {
        "task": "language_and_word_type_screen",
        "allowed_lexical_classes": [
            "shona_content", "shona_noncontent", "english_or_code_switch",
            "name_place_or_brand", "uncertain"
        ],
        "allowed_word_types": [
            "noun", "verb", "adjective", "adverb", "interjection", "idiom",
            "pronoun", "deictic", "demonstrative", "function", "particle",
            "conjunction", "preposition", "concord", "auxiliary", "name",
            "place", "identity", "unknown"
        ],
        "items": [item],
    }
    response_obj, cache_hit = call_rag_json(
        "candidate_recovery",
        RAG_RECOVERY_PROMPT_VERSION,
        recovery_system,
        payload,
    )
    recovery_cache_hits += int(cache_hit)
    recovery_api_calls += int(not cache_hit)
    returned = response_obj.get("items", [])
    if returned:
        recovery_results[token] = returned[0]

missing_recovery_results = sorted(
    set(item["token"] for item in recovery_input_items) - set(recovery_results)
)
if missing_recovery_results:
    raise RuntimeError(
        "Contextual recovery did not return results for: "
        + ", ".join(missing_recovery_results[:20])
    )

allowed_classes = {
    "shona_content", "shona_noncontent", "english_or_code_switch",
    "name_place_or_brand", "uncertain"
}
recovery_rows = []
for _, row in rag_recovery_pool.iterrows():
    token = row["token"]
    obj = recovery_results[token]
    lexical_class = normalize_text(obj.get("lexical_class", "uncertain")).replace(" ", "_")
    if lexical_class not in allowed_classes:
        lexical_class = "uncertain"
    confidence = normalise_confidence(obj.get("confidence", 0.0))
    recovered = bool(
        lexical_class in RAG_RECOVERY_ALLOWED_CLASSES
        and confidence >= RAG_RECOVERY_MIN_CONFIDENCE
    )
    recovery_rows.append({
        "token": token,
        "count": int(row["count"]),
        "document_support": int(row["document_support"]),
        "original_noise_reason": row["noise_reason"],
        "contextual_lexical_class": lexical_class,
        "contextual_word_type": str(obj.get("word_type", "") or "").strip(),
        "contextual_english_meaning": str(obj.get("english_meaning", "") or "").strip(),
        "contextual_confidence": confidence,
        "contextual_reason": str(obj.get("reason", "") or "").strip(),
        "recovered_as_shona_content": recovered,
        "assessment_model": RAG_MODEL,
        "assessment_prompt_version": RAG_RECOVERY_PROMPT_VERSION,
    })

rag_recovery = pd.DataFrame(recovery_rows)
if rag_recovery.empty:
    rag_recovery = pd.DataFrame(columns=[
        "token", "count", "document_support", "original_noise_reason",
        "contextual_lexical_class", "contextual_word_type",
        "contextual_english_meaning", "contextual_confidence",
        "contextual_reason", "recovered_as_shona_content",
        "assessment_model", "assessment_prompt_version"
    ])
rag_recovery.to_csv(TAB / "rag_candidate_recovery.csv", index=False)
rag_recovery.to_csv(TAB / "table_rag_candidate_recovery.csv", index=False)

recovered_tokens = set(
    rag_recovery.loc[
        rag_recovery["recovered_as_shona_content"].eq(True), "token"
    ]
)
rag_candidates = vocab[vocab["token"].isin(recovered_tokens)].copy()
rag_candidates["candidate_source"] = "rag_recovered"

def extra_filter_reason(token, candidate_source):
    if token in quarantined_only_forms:
        return "quarantined_inherited_score_form"
    if token in lex_forms:
        return "already_in_base_lexicon"
    # A contextual recovery decision is allowed to override the English-lexicon screen.
    if candidate_source != "rag_recovered" and token in lex_english:
        return "matches_english_lexicon"
    return ""

candidate_union = pd.concat(
    [deterministic_candidates, rag_candidates], ignore_index=True, sort=False
).sort_values(
    ["token", "candidate_source"], kind="mergesort"
).drop_duplicates(
    subset=["token"], keep="first"
).reset_index(drop=True)

candidate_union["extra_filter_reason"] = [
    extra_filter_reason(token, source)
    for token, source in zip(candidate_union["token"], candidate_union["candidate_source"])
]
candidate_union["candidate_status"] = np.where(
    candidate_union["extra_filter_reason"].eq(""),
    "candidate_missing_from_base_lexicon",
    candidate_union["extra_filter_reason"]
)
candidate_union.to_csv(TAB / "candidate_filter_audit.csv", index=False)

missing_candidates = candidate_union[
    candidate_union["candidate_status"].eq("candidate_missing_from_base_lexicon")
].copy()
missing_candidates.to_csv(TAB / "shona_candidates_vocab_v2.csv", index=False)

front_funnel = pd.DataFrame([
    ["Build tweets", len(train_work)],
    ["Token occurrences", int(vocab["count"].sum())],
    ["Raw unique tokens", len(vocab)],
    ["Deterministic likely-Shona candidates", len(deterministic_candidates)],
    ["Contextual recovery pool", len(rag_recovery_pool)],
    ["Contextually recovered Shona content tokens", len(recovered_tokens)],
    ["Combined likely-Shona candidates", len(candidate_union)],
    ["Already represented in clean base",
     int(candidate_union["candidate_status"].eq("already_in_base_lexicon").sum())],
    ["Missing from clean base", len(missing_candidates)],
], columns=["stage", "count"])
front_funnel.to_csv(TAB / "candidate_discovery_funnel.csv", index=False)
front_funnel.to_csv(TAB / "table_candidate_discovery_funnel.csv", index=False)

rag_metadata = pd.DataFrame([{
    "model": RAG_MODEL,
    "candidate_recovery_prompt_version": RAG_RECOVERY_PROMPT_VERSION,
    "candidate_recovery_prompt_sha256": hashlib.sha256(
        (recovery_system + RAG_RECOVERY_PROMPT_VERSION).encode("utf-8")
    ).hexdigest(),
    "screened_tokens": len(rag_recovery_pool),
    "recovered_tokens": len(recovered_tokens),
    "minimum_confidence": RAG_RECOVERY_MIN_CONFIDENCE,
    "api_calls": recovery_api_calls,
    "cache_hits": recovery_cache_hits,
    "data_source": "build_split_only",
}])
rag_metadata.to_csv(TAB / "rag_run_metadata.csv", index=False)

print(front_funnel.to_string(index=False))

plot_df = front_funnel[
    front_funnel["stage"].isin([
        "Raw unique tokens", "Deterministic likely-Shona candidates",
        "Contextually recovered Shona content tokens",
        "Combined likely-Shona candidates", "Missing from clean base"
    ])
].copy()
plt.figure(figsize=(9, 5.5))
bars = plt.bar(plot_df["stage"], plot_df["count"])
plt.ylabel("Unique tokens")
plt.title("Candidate discovery and contextual recovery")
plt.xticks(rotation=18, ha="right")
for bar, value in zip(bars, plot_df["count"]):
    plt.text(bar.get_x() + bar.get_width()/2, value, f"{int(value):,}",
             ha="center", va="bottom")
plt.tight_layout()
save_current_figure("fig_candidate_discovery_rag_recovery.png")

if len(rag_recovery):
    recovery_counts = rag_recovery["contextual_lexical_class"].value_counts().sort_values()
    plt.figure(figsize=(8, 4.8))
    bars = plt.barh(recovery_counts.index, recovery_counts.values)
    plt.xlabel("Tokens")
    plt.title("Contextual recovery decisions")
    for bar, value in zip(bars, recovery_counts.values):
        plt.text(value, bar.get_y() + bar.get_height()/2, f" {int(value):,}", va="center")
    plt.tight_layout()
    save_current_figure("fig_rag_candidate_recovery_decisions.png")


In [ ]:

# Build-only PMI, Fisher exact tests, FDR control and bootstrap effect intervals

enforce_train_work_boundary("before_pmi")

candidate_list = sorted(set(missing_candidates["token"]))
CANDIDATE_SOURCE = dict(zip(missing_candidates["token"], missing_candidates["candidate_source"]))
candidate_set = set(candidate_list)
document_counts = {word: Counter() for word in candidate_list}

for tokens, label in zip(train_work["_tokens"], train_work["_label"]):
    for word in set(tokens) & candidate_set:
        document_counts[word][label] += 1

N = len(train_work)
class_n = train_work["_label"].value_counts().to_dict()
for label in ["NEG", "NEU", "POS"]:
    class_n.setdefault(label, 0)

support_audit_rows = []
for word in candidate_list:
    counts = document_counts[word]
    neg = int(counts.get("NEG", 0)); neu = int(counts.get("NEU", 0)); pos = int(counts.get("POS", 0))
    support = neg + neu + pos
    support_audit_rows.append({
        "word": word, "review_flag": REVIEW_FLAGS.get(word, ""),
        "candidate_source": CANDIDATE_SOURCE.get(word, "deterministic"),
        "df_total": support, "neg_count": neg, "neu_count": neu, "pos_count": pos,
        "enters_statistical_testing": bool(support >= MIN_DOC_SUPPORT),
        "support_decision": "tested" if support >= MIN_DOC_SUPPORT else "below_minimum_document_support",
    })
pd.DataFrame(support_audit_rows).sort_values(
    ["enters_statistical_testing", "df_total", "word"], ascending=[False, False, True]
).to_csv(TAB / "candidate_document_support_audit.csv", index=False)

rng = np.random.default_rng(SEED)
alpha = 1 - CI_LEVEL
rows = []


def log_word_given_class(count, class_total):
    return float(np.log2((count + 0.5) / (class_total + 1.0)))


def log_word_marginal(support):
    return float(np.log2((support + 0.5) / (N + 1.0)))


def pmi_value(count, support, class_total):
    return log_word_given_class(count, class_total) - log_word_marginal(support)


def delta_from_counts(pos_present, neg_present, pos_total, neg_total):
    return np.log2((pos_present + 0.5) / (pos_total + 1.0)) - np.log2((neg_present + 0.5) / (neg_total + 1.0))


for word in candidate_list:
    counts = document_counts[word]
    neg = int(counts.get("NEG", 0)); neu = int(counts.get("NEU", 0)); pos = int(counts.get("POS", 0))
    support = neg + neu + pos
    if support < MIN_DOC_SUPPORT:
        continue

    pmi_neg = pmi_value(neg, support, class_n["NEG"])
    pmi_neu = pmi_value(neu, support, class_n["NEU"])
    pmi_pos = pmi_value(pos, support, class_n["POS"])
    delta = float(delta_from_counts(pos, neg, class_n["POS"], class_n["NEG"]))
    direction = "POS" if delta > 0 else "NEG" if delta < 0 else "NONE"

    # Two-sided Fisher exact test on word presence x POS/NEG class. This produces a proper p-value
    # for Benjamini–Hochberg correction; the bootstrap below is retained for the effect interval.
    contingency = np.array([
        [pos, class_n["POS"] - pos],
        [neg, class_n["NEG"] - neg],
    ], dtype=int)
    fisher_result = fisher_exact(contingency, alternative="two-sided")
    fisher_odds_ratio = float(fisher_result.statistic)
    fisher_p_value = float(fisher_result.pvalue)

    # Multinomial bootstrap over the six empirical presence/absence x class cells. This is the
    # per-word equivalent of resampling training tweets for the PMI difference.
    six_counts = np.array([
        neg, neu, pos,
        class_n["NEG"] - neg,
        class_n["NEU"] - neu,
        class_n["POS"] - pos,
    ], dtype=float)
    draws = rng.multinomial(N, six_counts / six_counts.sum(), size=BOOTSTRAP_REPS)
    boot_delta = delta_from_counts(
        draws[:, 2], draws[:, 0],
        draws[:, 2] + draws[:, 5],
        draws[:, 0] + draws[:, 3]
    )
    ci_low, ci_high = np.quantile(boot_delta, [alpha / 2, 1 - alpha / 2])
    direction_stability = (
        float((boot_delta > 0).mean()) if direction == "POS"
        else float((boot_delta < 0).mean()) if direction == "NEG" else 0.0
    )
    ci_excludes_zero = bool((ci_low > 0) or (ci_high < 0))

    dominant_count = pos if direction == "POS" else neg
    wilson_low, wilson_high = wilson_interval(dominant_count, support, CI_LEVEL)
    score = int(np.clip(round(9 * ((pos / support) - (neg / support))), -9, 9))

    rows.append({
        "word": word,
        "review_flag": REVIEW_FLAGS.get(word, ""),
        "candidate_source": CANDIDATE_SOURCE.get(word, "deterministic"),
        "df_total": support,
        "neg_count": neg, "neu_count": neu, "pos_count": pos,
        "pmi_neg": pmi_neg, "pmi_neu": pmi_neu, "pmi_pos": pmi_pos,
        "delta_pmi_pos_minus_neg": delta,
        "fisher_odds_ratio": fisher_odds_ratio,
        "fisher_p_value": fisher_p_value,
        "bootstrap_ci_low": float(ci_low), "bootstrap_ci_high": float(ci_high),
        "bootstrap_ci_excludes_zero": ci_excludes_zero,
        "direction": direction,
        "direction_stability": direction_stability,
        "dominant_share": dominant_count / support,
        "wilson_lower": float(wilson_low), "wilson_upper": float(wilson_high),
        "score_minus9_to9": score,
    })

pmi_all = pd.DataFrame(rows)

if len(pmi_all):
    survives_fdr, adjusted = benjamini_hochberg(pmi_all["fisher_p_value"].to_numpy(), FDR_ALPHA)
    pmi_all["bh_adjusted_p_value"] = adjusted
    pmi_all["survives_fdr"] = survives_fdr
    pmi_all["statistical_keep"] = (
        pmi_all["survives_fdr"]
        & pmi_all["bootstrap_ci_excludes_zero"]
        & (pmi_all["wilson_lower"] > MIN_WILSON_LOWER)
        & pmi_all["direction"].isin(["POS", "NEG"])
    )

    def statistical_reasons(row):
        reasons = []
        if not bool(row["survives_fdr"]):
            reasons.append("fails_bh_fdr")
        if not bool(row["bootstrap_ci_excludes_zero"]):
            reasons.append("bootstrap_ci_includes_zero")
        if not float(row["wilson_lower"]) > MIN_WILSON_LOWER:
            reasons.append("wilson_lower_not_above_threshold")
        if row["direction"] not in {"POS", "NEG"}:
            reasons.append("no_positive_or_negative_direction")
        return ";".join(reasons)

    pmi_all["statistical_reject_reasons"] = pmi_all.apply(statistical_reasons, axis=1)
    pmi_all = pmi_all.sort_values(
        ["statistical_keep", "bh_adjusted_p_value", "df_total"],
        ascending=[False, True, False]
    ).reset_index(drop=True)
else:
    pmi_all = pd.DataFrame(columns=[
        "word", "review_flag", "candidate_source", "df_total", "neg_count", "neu_count", "pos_count",
        "pmi_neg", "pmi_neu", "pmi_pos", "delta_pmi_pos_minus_neg",
        "fisher_odds_ratio", "fisher_p_value", "bh_adjusted_p_value", "survives_fdr",
        "bootstrap_ci_low", "bootstrap_ci_high", "bootstrap_ci_excludes_zero",
        "direction", "direction_stability", "dominant_share", "wilson_lower", "wilson_upper",
        "score_minus9_to9", "statistical_keep", "statistical_reject_reasons"
    ])

shortlist = pmi_all[pmi_all["statistical_keep"].eq(True)].copy().reset_index(drop=True)
shortlist.insert(0, "candidate_id", [f"C{i:03d}" for i in range(1, len(shortlist) + 1)])

# Candidate-discovery ablation: deterministic preprocessing alone versus
# deterministic preprocessing plus contextual recovery. BH is recalculated within each family.
def statistical_keep_for_family(frame):
    if frame.empty:
        return pd.Series(dtype=bool), pd.Series(dtype=float)
    keep_fdr, adjusted_p = benjamini_hochberg(frame["fisher_p_value"].to_numpy(), FDR_ALPHA)
    keep = (
        pd.Series(keep_fdr, index=frame.index)
        & frame["bootstrap_ci_excludes_zero"].astype(bool)
        & (frame["wilson_lower"].astype(float) > MIN_WILSON_LOWER)
        & frame["direction"].isin(["POS", "NEG"])
    )
    return keep, pd.Series(adjusted_p, index=frame.index)

deterministic_stat = pmi_all[pmi_all["candidate_source"].eq("deterministic")].copy()
deterministic_keep, deterministic_adjusted = statistical_keep_for_family(deterministic_stat)
if len(deterministic_stat):
    deterministic_stat["ablation_bh_adjusted_p_value"] = deterministic_adjusted
    deterministic_stat["ablation_statistical_keep"] = deterministic_keep
else:
    deterministic_stat["ablation_bh_adjusted_p_value"] = pd.Series(dtype=float)
    deterministic_stat["ablation_statistical_keep"] = pd.Series(dtype=bool)

ablation_table = pd.DataFrame([
    {
        "candidate_discovery_channel": "deterministic_only",
        "missing_candidate_pool": int(missing_candidates["candidate_source"].eq("deterministic").sum()),
        "support_eligible_candidates": int(len(deterministic_stat)),
        "after_bh_fdr": int((deterministic_stat["ablation_bh_adjusted_p_value"] <= FDR_ALPHA).sum()) if len(deterministic_stat) else 0,
        "final_statistical_shortlist": int(deterministic_keep.sum()) if len(deterministic_stat) else 0,
        "rag_recovered_candidates_in_shortlist": 0,
    },
    {
        "candidate_discovery_channel": "deterministic_plus_rag_recovery",
        "missing_candidate_pool": int(len(missing_candidates)),
        "support_eligible_candidates": int(len(pmi_all)),
        "after_bh_fdr": int(pmi_all["survives_fdr"].sum()) if len(pmi_all) else 0,
        "final_statistical_shortlist": int(pmi_all["statistical_keep"].sum()) if len(pmi_all) else 0,
        "rag_recovered_candidates_in_shortlist": int((pmi_all["statistical_keep"].astype(bool) & pmi_all["candidate_source"].eq("rag_recovered")).sum()) if len(pmi_all) else 0,
    },
])
ablation_table.to_csv(TAB / "candidate_discovery_ablation.csv", index=False)
deterministic_stat.to_csv(TAB / "deterministic_only_statistical_ablation.csv", index=False)

plt.figure(figsize=(8, 4.8))
x = np.arange(len(ablation_table))
plt.bar(x - 0.18, ablation_table["support_eligible_candidates"], width=0.36, label="Support eligible")
plt.bar(x + 0.18, ablation_table["final_statistical_shortlist"], width=0.36, label="Final shortlist")
plt.xticks(x, ["Deterministic only", "Deterministic + RAG"])
plt.ylabel("Candidate words")
plt.title("Candidate-discovery ablation")
plt.legend()
plt.tight_layout()
save_current_figure("fig_candidate_discovery_ablation.png")

pmi_all.to_csv(TAB / "pmi_all_candidates_v2.csv", index=False)
shortlist.to_csv(TAB / "pmi_shortlist_v2.csv", index=False)

support_sensitivity = pd.DataFrame({
    "minimum_support": [5, 10, 15, 20],
    "candidate_words_at_or_above_support": [
        sum(sum(document_counts[word].values()) >= threshold for word in candidate_list)
        for threshold in [5, 10, 15, 20]
    ]
})
support_sensitivity.to_csv(AUD / "minimum_support_sensitivity_descriptive_only.csv", index=False)

threshold_record = {
    "construction_data": "fixed build split from strict deduplicated ShonaSenti training data only",
    "minimum_document_support": MIN_DOC_SUPPORT,
    "fisher_test": "two-sided Fisher exact test on word presence by POS/NEG class",
    "false_discovery_rate": FDR_ALPHA,
    "multiple_testing_correction": "Benjamini-Hochberg over all tested candidates",
    "bootstrap_replicates": BOOTSTRAP_REPS,
    "bootstrap_interval": CI_LEVEL,
    "wilson_lower_bound_required_above": MIN_WILSON_LOWER,
    "wilson_numerator": "selected POS or NEG build-tweet presence count",
    "wilson_denominator": "all build tweets containing the word: NEG + NEU + POS",
    "selection_rule": (
        "support >= 10 unique training tweets; Fisher q <= 0.05 after BH correction; "
        "95% bootstrap CI for POS-NEG PMI excludes zero; the 95% Wilson lower bound "
        "for the selected POS or NEG count divided by all build tweets containing the word "
        "(NEG + NEU + POS) exceeds 0.50"
    ),
    "direction_stability": "reported as a diagnostic, not a separate gate",
    "test_data_used_to_choose_rule": False,
    "candidates_tested": int(len(pmi_all)),
    "statistical_shortlist": int(len(shortlist)),
}
write_json(AUD / "fixed_threshold_rule.json", threshold_record)

print("Candidates tested at support >=", MIN_DOC_SUPPORT, ":", len(pmi_all))
print("Surviving BH FDR:", int(pmi_all["survives_fdr"].sum()) if len(pmi_all) else 0)
print("Statistical shortlist:", len(shortlist))
if len(shortlist):
    print("The detailed statistical shortlist was saved to tables/pmi_shortlist_v2.csv.")
    print("Use the separate blinded review package before inspecting polarity columns.")

statistical_funnel = pd.DataFrame({
    "stage": [
        f"Missing candidates with support >= {MIN_DOC_SUPPORT}",
        "After BH FDR",
        "Effect CI excludes 0",
        "Final statistical shortlist",
    ],
    "count": [
        len(pmi_all),
        int(pmi_all["survives_fdr"].sum()) if len(pmi_all) else 0,
        int((pmi_all["survives_fdr"] & pmi_all["bootstrap_ci_excludes_zero"]).sum()) if len(pmi_all) else 0,
        len(shortlist),
    ]
})
full_funnel = pd.concat([front_funnel, statistical_funnel], ignore_index=True)
full_funnel.to_csv(TAB / "candidate_funnel_v2.csv", index=False)
full_funnel.to_csv(TAB / "table_candidate_funnel_full.csv", index=False)
funnel = statistical_funnel.rename(columns={"stage": "Stage", "count": "Candidates"})
plt.figure(figsize=(8, 4.5))
bars = plt.bar(funnel["Stage"], funnel["Candidates"])
plt.ylabel("Candidate words")
plt.title("Training-only candidate funnel with false-discovery control")
plt.xticks(rotation=15, ha="right")
for bar, value in zip(bars, funnel["Candidates"]):
    plt.text(bar.get_x() + bar.get_width()/2, value, f"{int(value):,}", ha="center", va="bottom")
plt.tight_layout()
save_current_figure("fig_candidate_funnel_fdr.png")

polarity_counts = shortlist["direction"].value_counts().reindex(["NEG", "POS"]).fillna(0)
plt.figure(figsize=(6, 4))
bars = plt.bar(polarity_counts.index, polarity_counts.values)
plt.ylabel("Candidate words")
plt.title("Statistically retained candidates by polarity")
for bar, value in zip(bars, polarity_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, value, f"{int(value)}", ha="center", va="bottom")
plt.tight_layout()
save_current_figure(["fig_pmi_shortlist_pos_neg_counts.png", "fig_pmi_41_validation_counts.png"])


In [ ]:
# Retrieve build-only contexts, run the contextual assessment and prepare the structured researcher review

enforce_train_work_boundary("before_context_retrieval")

context_rows = []
review_item_rows = []
context_assessment_input = []

for _, row in shortlist.iterrows():
    word = row["word"]
    candidate_id = row["candidate_id"]
    mask = train_work["_tokens"].apply(lambda words: word in set(words))
    hits = train_work.loc[mask, [TEXT_COL, "_label", "_clean"]].copy()

    selected_indices = []
    for label in ["NEG", "NEU", "POS"]:
        part = hits[hits["_label"].eq(label)]
        if len(part):
            selected_indices.append(part.sample(1, random_state=SEED + len(selected_indices)).index[0])
    if len(selected_indices) < 3:
        remaining = hits.index.difference(selected_indices)
        extra_n = min(3 - len(selected_indices), len(remaining))
        if extra_n:
            selected_indices.extend(hits.loc[remaining].sample(extra_n, random_state=SEED).index.tolist())

    chosen = hits.loc[selected_indices].copy() if selected_indices else hits.head(0)
    contexts_for_review = chosen["_clean"].astype(str).tolist()[:3]
    np.random.default_rng(SEED + int(candidate_id[1:])).shuffle(contexts_for_review)

    for _, hit in hits.iterrows():
        context_rows.append({
            "candidate_id": candidate_id,
            "word": word,
            "label_internal_not_for_review": hit["_label"],
            "training_context": hit["_clean"],
        })

    review_item_rows.append({
        "candidate_id": candidate_id,
        "word": word,
        "context_1": contexts_for_review[0] if len(contexts_for_review) > 0 else "",
        "context_2": contexts_for_review[1] if len(contexts_for_review) > 1 else "",
        "context_3": contexts_for_review[2] if len(contexts_for_review) > 2 else "",
    })

    counts = hits["_label"].value_counts().reindex(["NEG", "NEU", "POS"]).fillna(0).astype(int)
    labelled_examples = []
    for label in ["NEG", "NEU", "POS"]:
        for text in hits[hits["_label"].eq(label)]["_clean"].head(2):
            labelled_examples.append({"label": label, "text": str(text)})

    context_assessment_input.append({
        "candidate_id": candidate_id,
        "word": word,
        "candidate_source": row.get("candidate_source", "deterministic"),
        "neg_count": int(counts["NEG"]),
        "neu_count": int(counts["NEU"]),
        "pos_count": int(counts["POS"]),
        "corpus_direction": row["direction"],
        "corpus_score_minus9_to9": int(row["score_minus9_to9"]),
        "contexts": labelled_examples[:RAG_MAX_CONTEXTS_PER_TOKEN],
    })

contexts_internal = pd.DataFrame(context_rows)
review_items = pd.DataFrame(review_item_rows)
contexts_internal.to_csv(REV / "candidate_training_contexts_internal.csv", index=False)
review_items.to_csv(REV / "single_researcher_review_items.csv", index=False)

context_system = (
    "Assess possible Shona sentiment-lexicon candidates using only the supplied labelled "
    "contexts from the clean build split. Distinguish standalone lexical sentiment from "
    "observed corpus-associated sentiment. Return one complete result for every input item."
)

def context_payload(items):
    return {
        "task": "complete_contextual_lexical_assessment",
        "rules": [
            "Judge Shona validity and word type conservatively.",
            "Separate standalone lexical sentiment from corpus association.",
            "Use Context-dependent when lexical polarity changes with context.",
            "Use Not-applicable for a non-Shona lexical item.",
            "POS association requires a positive score and NEG association a negative score.",
            "NEU or No-stable-association requires score 0.",
            "Return non-empty meaning and reason fields.",
        ],
        "items": items,
    }

context_results = {}
context_errors = []
context_cache_hits = 0
context_api_calls = 0

for batch in batch_items(context_assessment_input, RAG_CONTEXT_BATCH_SIZE):
    response_obj, cache_hit = call_rag_json_schema(
        "context_assessment_complete", RAG_CONTEXT_PROMPT_VERSION,
        context_system, context_payload(batch),
        "context_assessment_complete_response", RAG_CONTEXT_RESPONSE_SCHEMA,
    )
    context_cache_hits += int(cache_hit)
    context_api_calls += int(not cache_hit)
    expected = {item["candidate_id"]: normalize_text(item["word"]) for item in batch}
    returned_ids = set()
    for result in response_obj.get("items", []):
        candidate_id = str(result.get("candidate_id", "") or "").strip().upper()
        if candidate_id not in expected or candidate_id in returned_ids:
            continue
        returned_ids.add(candidate_id)
        problems = validate_context_item(result, candidate_id, expected[candidate_id])
        if not problems:
            context_results[candidate_id] = result
        else:
            context_errors.append({"candidate_id": candidate_id, "attempt_scope": "batch", "problems": ";".join(problems)})

for item in context_assessment_input:
    candidate_id = item["candidate_id"]
    if candidate_id in context_results:
        continue
    last_problems = ["missing_from_batch_response"]
    for retry_index in range(1, RAG_CONTEXT_COMPLETION_RETRIES + 1):
        retry_item = dict(item)
        retry_item["completion_retry"] = retry_index
        response_obj, cache_hit = call_rag_json_schema(
            "context_assessment_complete_retry", RAG_CONTEXT_PROMPT_VERSION,
            context_system, context_payload([retry_item]),
            "context_assessment_complete_response", RAG_CONTEXT_RESPONSE_SCHEMA,
        )
        context_cache_hits += int(cache_hit)
        context_api_calls += int(not cache_hit)
        returned = response_obj.get("items", [])
        if not returned:
            last_problems = ["empty_retry_response"]
            continue
        problems = validate_context_item(returned[0], candidate_id, normalize_text(item["word"]))
        if not problems:
            context_results[candidate_id] = returned[0]
            break
        last_problems = problems
    if candidate_id not in context_results:
        context_errors.append({"candidate_id": candidate_id, "attempt_scope": "individual_retry_exhausted", "problems": ";".join(last_problems)})

missing_context_results = sorted(set(item["candidate_id"] for item in context_assessment_input) - set(context_results))
if context_errors:
    pd.DataFrame(context_errors).to_csv(AUD / "rag_contextual_assessment_validation_audit.csv", index=False)
if missing_context_results:
    pd.DataFrame({"candidate_id": missing_context_results}).to_csv(AUD / "rag_contextual_assessment_incomplete.csv", index=False)
    raise RuntimeError("Contextual assessment remained incomplete after structured retries.")

assessment_rows = []
for item in context_assessment_input:
    obj = context_results[item["candidate_id"]]
    assessment_rows.append({
        "candidate_id": item["candidate_id"],
        "word": item["word"],
        "candidate_source": item["candidate_source"],
        "neg_contexts": item["neg_count"],
        "neu_contexts": item["neu_count"],
        "pos_contexts": item["pos_count"],
        "statistical_corpus_direction": item["corpus_direction"],
        "statistical_corpus_score_minus9_to9": item["corpus_score_minus9_to9"],
        "contextual_valid_shona_proposal": obj["valid_shona_proposal"],
        "contextual_english_meaning": obj["english_meaning"].strip(),
        "contextual_word_type": obj["word_type"],
        "contextual_lexical_sentiment": obj["lexical_sentiment"],
        "contextual_corpus_association": obj["corpus_association"],
        "contextual_sentiment_basis_proposal": obj["sentiment_basis_proposal"],
        "contextual_confidence": float(obj["confidence"]),
        "contextual_provisional_score_minus9_to9": int(obj["provisional_score_minus9_to9"]),
        "contextual_corpus_direction_agrees_statistics": obj["corpus_association"] == item["corpus_direction"],
        "contextual_lexical_status": obj["lexical_status"],
        "contextual_reason": obj["reason"].strip(),
        "assessment_complete": True,
        "assessment_model": RAG_MODEL,
        "assessment_prompt_version": RAG_CONTEXT_PROMPT_VERSION,
    })

rag_context_assessment = pd.DataFrame(assessment_rows)
rag_context_assessment.to_csv(TAB / "rag_contextual_candidate_assessment_complete.csv", index=False)

review_template = review_items[["candidate_id", "word"]].copy()
review_template.insert(0, "reviewer_id", RESEARCHER_REVIEWER_ID)
for column in ["shona_level", "valid_shona", "meaning", "sentiment", "confidence", "word_type", "comment"]:
    review_template[column] = ""
review_template.to_csv(REV / "single_researcher_review_template.csv", index=False)

review_field_guide = pd.DataFrame([
    ["reviewer_id", "RESEARCHER", "Required"],
    ["shona_level", "Native | L1 | First language | Mother tongue | Fluent", "Required"],
    ["valid_shona", "Yes | No | Unsure", "Required"],
    ["meaning", "Free-text English meaning", "Required"],
    ["sentiment", "NEG | NEU | POS | Context-dependent", "Required when valid Shona"],
    ["confidence", "Low | Medium | High", "Required"],
    ["word_type", "noun | verb | adjective | adverb | interjection | idiom | phrase | pronoun | deictic | demonstrative | function | particle | conjunction | preposition | concord | auxiliary | name | place | identity", "Required when valid Shona"],
    ["comment", "Free text", "Optional"],
], columns=["field", "allowed_value_or_description", "requirement"])
review_field_guide.to_csv(REV / "single_researcher_review_field_guide.csv", index=False)

review_zip_path = ROOT_OUT.parent / "FrenchyShona_V2_single_researcher_review_package_v7_4.zip"
review_zip_temp = ROOT_OUT.parent / ".FrenchyShona_V2_single_researcher_review_package_v7_4.zip.tmp"
if review_zip_temp.exists():
    review_zip_temp.unlink()
with zipfile.ZipFile(review_zip_temp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for review_file in [REV / "single_researcher_review_items.csv", REV / "single_researcher_review_template.csv", REV / "single_researcher_review_field_guide.csv"]:
        archive.write(review_file, arcname=review_file.name)
with zipfile.ZipFile(review_zip_temp, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Review-package ZIP failed at {bad_member}")
os.replace(review_zip_temp, review_zip_path)

pd.DataFrame([{
    "model": RAG_MODEL,
    "context_prompt_version": RAG_CONTEXT_PROMPT_VERSION,
    "assessed_candidates": len(rag_context_assessment),
    "complete_assessments": int(rag_context_assessment["assessment_complete"].sum()),
    "api_calls": context_api_calls,
    "cache_hits": context_cache_hits,
    "data_source": "build_split_only",
}]).to_csv(TAB / "rag_context_run_metadata.csv", index=False)

with pd.ExcelWriter(TAB / "FrenchyShona_V2_candidate_evidence.xlsx", engine="openpyxl") as writer:
    front_funnel.to_excel(writer, sheet_name="candidate_discovery", index=False)
    ablation_table.to_excel(writer, sheet_name="discovery_ablation", index=False)
    rag_recovery.to_excel(writer, sheet_name="rag_recovery", index=False)
    candidate_union.to_excel(writer, sheet_name="combined_candidates", index=False)
    pmi_all.to_excel(writer, sheet_name="statistical_candidates", index=False)
    shortlist.to_excel(writer, sheet_name="statistical_shortlist", index=False)
    rag_context_assessment.to_excel(writer, sheet_name="contextual_assessment", index=False)
    review_items.to_excel(writer, sheet_name="review_items", index=False)
    review_template.to_excel(writer, sheet_name="review_template", index=False)

if len(rag_context_assessment):
    counts = rag_context_assessment.set_index("word")[["neg_contexts", "neu_contexts", "pos_contexts"]]
    counts.plot(kind="bar", stacked=True, figsize=(10, 5.5))
    plt.ylabel("Build-tweet occurrences")
    plt.title("Retrieved sentiment contexts for statistically retained candidates")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    save_current_figure("fig_rag_context_sentiment_distribution.png")

    status_counts = rag_context_assessment["contextual_lexical_status"].value_counts().sort_index()
    plt.figure(figsize=(6, 4))
    bars = plt.bar(status_counts.index, status_counts.values)
    plt.ylabel("Candidates")
    plt.title("Contextual assessment outcomes")
    for bar, value in zip(bars, status_counts.values):
        plt.text(bar.get_x() + bar.get_width()/2, value, str(int(value)), ha="center", va="bottom")
    plt.tight_layout()
    save_current_figure("fig_rag_contextual_assessment_status.png")

print("Single-researcher review items prepared:", len(review_items))
print("Complete contextual candidate assessments:", len(rag_context_assessment))
print("No researcher input is required during the first run.")


In [ ]:
# Validate the structured researcher review and combine the evidence streams

researcher_decisions = None
validated_additions = pd.DataFrame()
researcher_review_complete = (len(shortlist) == 0)
researcher_validation_summary = {
    "review_file_found": bool(REVIEW_PATH),
    "validation_design": VALIDATION_DESIGN,
    "interim_resource": INTERIM_RESOURCE,
    "required_reviewer_id": RESEARCHER_REVIEWER_ID,
    "shortlist_items": int(len(shortlist)),
    "inter_rater_agreement_applicable": False,
    "independent_native_speaker_validation": False,
}

def normalise_valid(value):
    text = normalize_text(value)
    if text in {"yes", "y", "valid", "shona", "1", "true"}: return "YES"
    if text in {"no", "n", "invalid", "0", "false"}: return "NO"
    return "UNSURE"

def normalise_sentiment(value):
    text = normalize_text(value).upper().replace("_", " ").replace("-", " ")
    mapping = {"POSITIVE": "POS", "NEGATIVE": "NEG", "NEUTRAL": "NEU", "CONTEXT DEPENDENT": "CONTEXT", "CONTEXTDEPENDENT": "CONTEXT", "N/A": "NOT_APPLICABLE", "NA": "NOT_APPLICABLE", "NOT APPLICABLE": "NOT_APPLICABLE"}
    text = mapping.get(text, text)
    return text if text in {"POS", "NEG", "NEU", "CONTEXT", "NOT_APPLICABLE"} else "UNSURE"

def normalise_confidence(value):
    text = normalize_text(value).upper()
    return text if text in {"LOW", "MEDIUM", "HIGH"} else "UNSURE"

NON_CONTENT_TYPES = {"PRONOUN", "DEICTIC", "DEMONSTRATIVE", "FUNCTION", "PARTICLE", "CONJUNCTION", "PREPOSITION", "CONCORD", "AUXILIARY", "NAME", "PLACE", "NUMBER"}
IDENTITY_TYPES = {"IDENTITY", "ETHNONYM", "DEMONYM", "NATIONALITY"}
CONTENT_TYPES = {"NOUN", "VERB", "ADJECTIVE", "ADVERB", "INTERJECTION", "IDIOM", "PHRASE"}
WORD_TYPE_MAP = {
    "PRONOUN": "PRONOUN", "PRONOUNS": "PRONOUN", "DEICTIC": "DEICTIC", "DEMONSTRATIVE": "DEMONSTRATIVE",
    "FUNCTION": "FUNCTION", "FUNCTION WORD": "FUNCTION", "FUNCTIONWORD": "FUNCTION", "GRAMMATICAL": "FUNCTION", "GRAMMAR": "FUNCTION",
    "PARTICLE": "PARTICLE", "DISCOURSE": "PARTICLE", "DISCOURSE MARKER": "PARTICLE", "CONJUNCTION": "CONJUNCTION",
    "PREPOSITION": "PREPOSITION", "CONCORD": "CONCORD", "AUXILIARY": "AUXILIARY", "NUMBER": "NUMBER",
    "NAME": "NAME", "PROPER NOUN": "NAME", "PERSON": "NAME", "SURNAME": "NAME", "BRAND": "NAME",
    "PLACE": "PLACE", "LOCATION": "PLACE", "IDENTITY": "IDENTITY", "IDENTITY TERM": "IDENTITY", "ETHNONYM": "ETHNONYM",
    "ETHNIC": "ETHNONYM", "DEMONYM": "DEMONYM", "NATIONALITY": "NATIONALITY", "NOUN": "NOUN", "VERB": "VERB",
    "ADJECTIVE": "ADJECTIVE", "ADJ": "ADJECTIVE", "ADVERB": "ADVERB", "INTERJECTION": "INTERJECTION",
    "IDIOM": "IDIOM", "PHRASE": "PHRASE", "EXPRESSION": "PHRASE", "N/A": "NOT_APPLICABLE", "NA": "NOT_APPLICABLE", "NOT APPLICABLE": "NOT_APPLICABLE",
}

def normalise_word_type(value):
    text = normalize_text(value).upper().replace("-", " ").replace("_", " ").strip()
    return WORD_TYPE_MAP.get(text, "UNSURE" if not text else "OTHER")

def normalise_shona_level(value):
    text = normalize_text(value).upper().replace("-", " ").replace("_", " ").strip()
    if "NATIVE" in text: return "NATIVE"
    if text in {"L1", "FIRST LANGUAGE", "FIRSTLANGUAGE"}: return "FIRST_LANGUAGE"
    if "MOTHER TONGUE" in text or "MOTHERTONGUE" in text: return "MOTHER_TONGUE"
    if "FLUENT" in text: return "FLUENT"
    return "OTHER"

def read_researcher_review(path):
    try:
        frame = pd.read_csv(path, sep=None, engine="python", keep_default_na=False, na_filter=False)
    except Exception:
        frame = pd.read_csv(path, keep_default_na=False, na_filter=False)
    frame.columns = [str(c).strip() for c in frame.columns]
    return frame

if REVIEW_PATH:
    reviews = read_researcher_review(REVIEW_PATH)
    required_cols = {"reviewer_id", "candidate_id", "word", "shona_level", "valid_shona", "meaning", "sentiment", "confidence", "word_type"}
    missing_cols = required_cols - set(reviews.columns)
    if missing_cols:
        raise ValueError(f"Researcher review is missing required columns: {sorted(missing_cols)}")
    if "comment" not in reviews.columns:
        reviews["comment"] = ""

    reviews["reviewer_id"] = reviews["reviewer_id"].astype("string").fillna("").str.strip().str.upper()
    reviews["candidate_id"] = reviews["candidate_id"].astype("string").fillna("").str.strip().str.upper()
    reviews["word"] = reviews["word"].map(normalize_text)
    reviews["shona_level_norm"] = reviews["shona_level"].map(normalise_shona_level)
    reviews["valid_norm"] = reviews["valid_shona"].map(normalise_valid)
    reviews["sentiment_norm"] = reviews["sentiment"].map(normalise_sentiment)
    reviews["confidence_norm"] = reviews["confidence"].map(normalise_confidence)
    reviews["word_type_norm"] = reviews["word_type"].map(normalise_word_type)
    reviews["meaning_nonblank"] = reviews["meaning"].astype("string").fillna("").str.strip().ne("")

    reviewer_ids = sorted(set(reviews["reviewer_id"]))
    if reviewer_ids != [RESEARCHER_REVIEWER_ID]:
        raise ValueError(f"Expected reviewer_id {RESEARCHER_REVIEWER_ID!r}; observed {reviewer_ids}.")
    if reviews.duplicated(["reviewer_id", "candidate_id"]).any():
        raise ValueError("Duplicate researcher/candidate rows found.")

    expected_pairs = set(zip(shortlist["candidate_id"], shortlist["word"].map(normalize_text)))
    observed_pairs = set(zip(reviews["candidate_id"], reviews["word"]))
    if observed_pairs != expected_pairs:
        pd.concat([pd.Series(sorted(expected_pairs), name="expected"), pd.Series(sorted(observed_pairs), name="observed")], axis=1).to_csv(REV / "researcher_review_candidate_set_mismatch.csv", index=False)
        raise ValueError("The review candidate set does not exactly match the current statistical shortlist.")

    review_file_hash = sha256_file(REVIEW_PATH)
    pd.DataFrame([{"review_file": str(REVIEW_PATH), "sha256": review_file_hash, "candidate_set_exact_match": True, "review_precedes_contextual_comparison": True}]).to_csv(REV / "researcher_review_provenance.csv", index=False)

    reviews["eligible_researcher"] = reviews["shona_level_norm"].isin(ELIGIBLE_SHONA_LEVELS)
    valid_yes = reviews["valid_norm"].eq("YES")
    valid_no = reviews["valid_norm"].eq("NO")
    valid_shona_specific_fields = reviews["sentiment_norm"].isin({"POS", "NEG", "NEU", "CONTEXT"}) & reviews["word_type_norm"].isin(CONTENT_TYPES | NON_CONTENT_TYPES | IDENTITY_TYPES)
    invalid_shona_allowed_fields = reviews["sentiment_norm"].isin({"POS", "NEG", "NEU", "CONTEXT", "NOT_APPLICABLE"}) & reviews["word_type_norm"].isin(CONTENT_TYPES | NON_CONTENT_TYPES | IDENTITY_TYPES | {"NOT_APPLICABLE"})
    reviews["complete_rating"] = reviews["eligible_researcher"] & reviews["reviewer_id"].eq(RESEARCHER_REVIEWER_ID) & (valid_yes | valid_no) & reviews["confidence_norm"].ne("UNSURE") & reviews["meaning_nonblank"] & ((valid_yes & valid_shona_specific_fields) | (valid_no & invalid_shona_allowed_fields))
    reviews.loc[~reviews["complete_rating"]].to_csv(REV / "single_researcher_incomplete_rows.csv", index=False)
    researcher_review_complete = bool(reviews["complete_rating"].all())

    researcher_validation_summary.update({"reviewer_id": RESEARCHER_REVIEWER_ID, "review_file_sha256": review_file_hash, "complete_ratings": int(reviews["complete_rating"].sum()), "all_shortlist_items_reviewed": researcher_review_complete})

    merged = shortlist.merge(reviews[["candidate_id", "word", "valid_norm", "meaning", "sentiment_norm", "confidence_norm", "word_type_norm", "comment"]], on=["candidate_id", "word"], how="left").merge(rag_context_assessment.drop(columns=["candidate_source"], errors="ignore"), on=["candidate_id", "word"], how="left", validate="one_to_one")

    decision_rows = []
    for _, row in merged.iterrows():
        valid_shona = row["valid_norm"]
        researcher_sentiment = row["sentiment_norm"]
        researcher_word_type = row["word_type_norm"]
        corpus_direction = row["direction"]
        corpus_score = int(row["score_minus9_to9"])
        researcher_content = researcher_word_type in CONTENT_TYPES
        researcher_noncontent = researcher_word_type in NON_CONTENT_TYPES
        researcher_identity = researcher_word_type in IDENTITY_TYPES

        contextual_complete = bool(row.get("assessment_complete", False))
        contextual_validity = normalize_text(
            row.get("contextual_valid_shona_proposal", "")
        )
        contextual_status = normalize_text(
            row.get("contextual_lexical_status", "")
        )
        contextual_association = str(
            row.get("contextual_corpus_association", "")
        ).strip().upper()

        try:
            contextual_confidence_value = float(
                row.get("contextual_confidence", np.nan)
            )
        except Exception:
            contextual_confidence_value = np.nan

        try:
            contextual_score_value = int(round(float(
                row.get("contextual_provisional_score_minus9_to9", 0)
            )))
        except Exception:
            contextual_score_value = 0

        contextual_validity_allows_corpus = (
            contextual_validity in {"yes", "uncertain"}
        )
        contextual_status_allows_corpus = (
            contextual_status
            in CONTEXTUAL_STATUSES_ALLOWED_FOR_CORPUS_ASSOCIATION
        )
        contextual_confidence_pass = bool(
            np.isfinite(contextual_confidence_value)
            and contextual_confidence_value >= RAG_CONTEXT_MIN_CONFIDENCE
        )
        contextual_association_agrees = bool(
            contextual_association == corpus_direction
        )
        contextual_score_direction_matches = bool(
            (corpus_direction == "POS" and contextual_score_value > 0)
            or (corpus_direction == "NEG" and contextual_score_value < 0)
        )

        contextual_supports_corpus = bool(
            contextual_complete
            and contextual_validity_allows_corpus
            and contextual_status_allows_corpus
            and contextual_confidence_pass
            and contextual_association_agrees
            and contextual_score_direction_matches
        )

        rejection_reasons = []
        if valid_shona == "NO": rejection_reasons.append("researcher_identified_non_shona")
        elif valid_shona != "YES": rejection_reasons.append("researcher_shona_validity_uncertain")
        if researcher_noncontent: rejection_reasons.append("researcher_non_content_word_type")
        if researcher_identity: rejection_reasons.append("identity_term_requires_future_multi_reviewer_validation")
        if not researcher_content and valid_shona == "YES": rejection_reasons.append("not_content_bearing_lexical_item")
        if corpus_score == 0:
            rejection_reasons.append("corpus_score_zero")

        if not contextual_complete:
            rejection_reasons.append("contextual_assessment_incomplete")
        else:
            if not contextual_validity_allows_corpus:
                rejection_reasons.append(
                    "contextual_validity_does_not_support_shona_candidate"
                )
            if not contextual_status_allows_corpus:
                rejection_reasons.append(
                    "contextual_status_rejects_corpus_association"
                )
            if not contextual_confidence_pass:
                rejection_reasons.append(
                    "contextual_confidence_below_threshold"
                )
            if not contextual_association_agrees:
                rejection_reasons.append(
                    "contextual_corpus_direction_mismatch"
                )
            if not contextual_score_direction_matches:
                rejection_reasons.append(
                    "contextual_score_direction_mismatch"
                )

        if not rejection_reasons:
            contextual_lexical_sentiment = str(
                row.get("contextual_lexical_sentiment", "")
            ).strip().upper()
            if (
                researcher_sentiment == corpus_direction
                and contextual_lexical_sentiment == corpus_direction
                and contextual_status == "support"
            ):
                sentiment_basis = "lexical_polarity"
            else:
                sentiment_basis = "corpus_association"
            final_status = "KEEP"
            final_sentiment = corpus_direction
            final_score = corpus_score
        else:
            sentiment_basis = "not_retained"
            final_status = "REJECT"
            final_sentiment = ""
            final_score = np.nan

        decision_rows.append({
            **row.to_dict(),
            "candidate_source": row.get("candidate_source", "deterministic"),
            "corpus_direction": corpus_direction,
            "corpus_score_minus9_to9": corpus_score,
            "researcher_lexical_sentiment": researcher_sentiment,
            "researcher_word_type": researcher_word_type,
            "researcher_valid_shona": valid_shona,
            "researcher_meaning": row["meaning"],
            "researcher_confidence": row["confidence_norm"],
            "researcher_comment": row.get("comment", ""),
            "final_decision_rule_version": FINAL_DECISION_RULE_VERSION,
            "contextual_assessment_status": contextual_status,
            "contextual_validity_allows_corpus_association": contextual_validity_allows_corpus,
            "contextual_status_allows_corpus_association": contextual_status_allows_corpus,
            "contextual_confidence_pass": contextual_confidence_pass,
            "contextual_association_agrees_statistics": contextual_association_agrees,
            "contextual_score_direction_matches_statistics": contextual_score_direction_matches,
            "contextual_supports_corpus_association": contextual_supports_corpus,
            "researcher_corpus_disagreement": bool(researcher_sentiment in {"POS", "NEG", "NEU", "CONTEXT"} and researcher_sentiment != corpus_direction),
            "contextual_corpus_agreement": bool(contextual_association == corpus_direction),
            "three_way_lexical_agreement": bool(researcher_sentiment in {"POS", "NEG"} and researcher_sentiment == corpus_direction and row.get("contextual_lexical_sentiment") == corpus_direction),
            "final_sentiment_basis": sentiment_basis,
            "final_sentiment": final_sentiment,
            "final_score_minus9_to9": final_score,
            "final_status": final_status,
            "final_decision_reason": ";".join(rejection_reasons),
        })

    researcher_decisions = pd.DataFrame(decision_rows)
    researcher_decisions.to_csv(TAB / "candidate_evidence_comparison.csv", index=False)
    researcher_decisions.to_csv(REV / "single_researcher_decisions.csv", index=False)
    reviews.to_csv(REV / "single_researcher_review_normalised.csv", index=False)
    write_json(REV / "single_researcher_validation_summary.json", researcher_validation_summary)
    validated_additions = researcher_decisions[researcher_decisions["final_status"].eq("KEEP")].copy()

    with pd.ExcelWriter(TAB / "FrenchyShona_V2_final_candidate_decisions.xlsx", engine="openpyxl") as writer:
        researcher_decisions.to_excel(writer, sheet_name="candidate_evidence", index=False)
        validated_additions.to_excel(writer, sheet_name="retained_additions", index=False)
        ablation_table.to_excel(writer, sheet_name="discovery_ablation", index=False)

    evidence_counts = pd.Series({
        "Researcher valid Shona": int(researcher_decisions["researcher_valid_shona"].eq("YES").sum()),
        "Researcher content word": int(researcher_decisions["researcher_word_type"].isin(CONTENT_TYPES).sum()),
        "Context supports corpus": int(researcher_decisions["contextual_supports_corpus_association"].sum()),
        "Final KEEP": int(researcher_decisions["final_status"].eq("KEEP").sum()),
    })
    plt.figure(figsize=(8, 4.5))
    bars = plt.bar(evidence_counts.index, evidence_counts.values)
    plt.ylabel("Candidates")
    plt.title("Candidate evidence and retention")
    plt.xticks(rotation=20, ha="right")
    for bar, value in zip(bars, evidence_counts.values):
        plt.text(bar.get_x() + bar.get_width()/2, value, str(int(value)), ha="center", va="bottom")
    plt.tight_layout()
    save_current_figure("fig_candidate_evidence_agreement.png")

    print(json_text(researcher_validation_summary))
    print("Single-researcher review complete:", researcher_review_complete)
    print("KEEP:", len(validated_additions), "REJECT:", int(researcher_decisions["final_status"].eq("REJECT").sum()))
else:
    write_json(REV / "single_researcher_validation_summary.json", researcher_validation_summary)
    print("No completed researcher review supplied. The run ends with the complete evidence and blinded review files.")


In [ ]:

# Freeze the evaluation resource before opening the strict unseen evaluation

frozen_lexicon = None
frozen_path = None
freeze_allowed = researcher_review_complete and (REVIEW_PATH is not None or len(shortlist) == 0)

if freeze_allowed:
    enforce_train_work_boundary("before_freeze")
    master_clean = master_lex.drop(columns=[c for c in ["_shona_norm", "_expanded_norm"] if c in master_lex.columns]).copy()
    new_fields = [
        "v2_provenance", "v2_candidate_id", "v2_validation_status", "v2_validation_design",
        "v2_sentiment_basis", "v2_lexical_sentiment", "v2_corpus_association",
        "v2_researcher_word_type", "v2_researcher_meaning", "v2_researcher_confidence",
        "v2_training_support", "v2_candidate_source", "v2_contextual_model",
        "v2_contextual_valid_shona", "v2_contextual_word_type",
        "v2_contextual_lexical_sentiment", "v2_contextual_corpus_association",
        "v2_contextual_confidence", "v2_contextual_status", "v2_contextual_reason", "v2_evidence_disagreement", "v2_final_decision_rule_version",
        "v2_bh_adjusted_p_value", "v2_bootstrap_ci_low", "v2_bootstrap_ci_high"
    ]
    for field in new_fields:
        if field not in master_clean.columns:
            master_clean[field] = pd.NA
    master_clean["v2_provenance"] = "inherited_clean_base"
    master_clean["v2_validation_status"] = "inherited_evaluation_ready"
    master_clean["v2_validation_design"] = VALIDATION_DESIGN

    score_out = find_col_ci(master_clean, ["Score"])
    sentiment_out = find_col_ci(master_clean, ["Sentiment"])
    shona_out = find_col_ci(master_clean, ["Shona"])
    expanded_out = find_col_ci(master_clean, ["expanded_shona"])
    expanded_class_out = find_col_ci(master_clean, ["expanded_shona_class"])
    master_columns = list(master_clean.columns)

    if len(validated_additions) and (
        validated_additions["candidate_id"].duplicated().any()
        or validated_additions["word"].duplicated().any()
    ):
        raise ValueError("Validated additions contain duplicate candidate IDs or duplicate words.")

    addition_rows = []
    for _, row in validated_additions.iterrows():
        record = {column: pd.NA for column in master_columns}
        support = int(row["df_total"])
        record[shona_out] = row["word"]
        record[expanded_out] = row["word"]
        if expanded_class_out:
            record[expanded_class_out] = "train_only_single_researcher_interim_addition"
        record[score_out] = int(row["final_score_minus9_to9"])
        if sentiment_out:
            record[sentiment_out] = "Positive" if row["final_sentiment"] == "POS" else "Negative"
        record["v2_provenance"] = "training_corpus_addition"
        record["v2_candidate_id"] = row["candidate_id"]
        record["v2_validation_status"] = "single_researcher_interim_keep"
        record["v2_validation_design"] = VALIDATION_DESIGN
        record["v2_sentiment_basis"] = row["final_sentiment_basis"]
        record["v2_lexical_sentiment"] = row["researcher_lexical_sentiment"]
        record["v2_corpus_association"] = row["corpus_direction"]
        record["v2_researcher_word_type"] = row["researcher_word_type"]
        record["v2_researcher_meaning"] = row["researcher_meaning"]
        record["v2_researcher_confidence"] = row["researcher_confidence"]
        record["v2_training_support"] = int(row["df_total"])
        record["v2_candidate_source"] = row.get("candidate_source", "")
        record["v2_contextual_model"] = row.get("assessment_model", "")
        record["v2_contextual_valid_shona"] = row.get("contextual_valid_shona_proposal", "")
        record["v2_contextual_word_type"] = row.get("contextual_word_type", "")
        record["v2_contextual_lexical_sentiment"] = row.get("contextual_lexical_sentiment", "")
        record["v2_contextual_corpus_association"] = row.get("contextual_corpus_association", "")
        record["v2_contextual_confidence"] = row.get("contextual_confidence", pd.NA)
        record["v2_contextual_status"] = row.get("contextual_assessment_status", "")
        record["v2_contextual_reason"] = row.get("contextual_reason", "")
        record["v2_final_decision_rule_version"] = FINAL_DECISION_RULE_VERSION
        record["v2_evidence_disagreement"] = bool(row.get("researcher_corpus_disagreement", False))
        record["v2_bh_adjusted_p_value"] = float(row["bh_adjusted_p_value"])
        record["v2_bootstrap_ci_low"] = float(row["bootstrap_ci_low"])
        record["v2_bootstrap_ci_high"] = float(row["bootstrap_ci_high"])
        addition_rows.append(record)

    additions_df = pd.DataFrame(addition_rows, columns=master_columns)
    existing_forms = set(master_lex["_shona_norm"]) | set(master_lex["_expanded_norm"])
    existing_forms.discard("")
    if len(additions_df):
        addition_norms = additions_df[shona_out].astype("string").fillna("").map(normalize_text)
        unexpected_existing = additions_df[addition_norms.isin(existing_forms)].copy()
        if len(unexpected_existing):
            unexpected_existing.to_csv(AUD / "validated_additions_already_in_base_error.csv", index=False)
            raise ValueError(
                "A validated addition is already represented in the 6,615-row evaluation-ready base. "
                "No rows were silently removed; inspect validated_additions_already_in_base_error.csv."
            )

    frozen_lexicon = pd.concat([master_clean, additions_df], ignore_index=True)
    if len(frozen_lexicon) != len(master_clean) + len(additions_df):
        raise AssertionError("Frozen row count does not equal base rows plus validated additions.")

    # Validate the complete resource before any frozen CSV or hash is written.
    raw_score_text = frozen_lexicon[score_out].astype("string").fillna("").str.strip()
    numeric_scores = pd.to_numeric(frozen_lexicon[score_out], errors="coerce")
    invalid_nonempty = raw_score_text.ne("") & numeric_scores.isna()
    out_of_range = numeric_scores.notna() & ((numeric_scores < -9) | (numeric_scores > 9))

    score_audit = {
        "score_min": float(numeric_scores.dropna().min()) if numeric_scores.notna().any() else None,
        "score_max": float(numeric_scores.dropna().max()) if numeric_scores.notna().any() else None,
        "non_numeric_nonblank_count": int(invalid_nonempty.sum()),
        "out_of_range_score_count": int(out_of_range.sum()),
        "clipping_is_identity_on_valid_score_range": bool(not out_of_range.any()),
        "reported_systems": [
            "sign-based aggregate score",
            "validation-calibrated average score",
        ],
        "note": (
            "A clipped variant is not reported as a separate system because the "
            "FrenchyLuba score schema is already restricted to [-9, 9]."
        ),
        "validation_timing": "before frozen CSV and SHA-256 creation",
    }
    write_json(AUD / "lexicon_score_range_and_clipping_audit.json", score_audit)

    if invalid_nonempty.any() or out_of_range.any():
        problem_rows = frozen_lexicon.loc[
            invalid_nonempty | out_of_range,
            [c for c in [shona_out, expanded_out, score_out] if c],
        ].copy()
        problem_rows.to_csv(AUD / "lexicon_score_validation_failures.csv", index=False)
        raise ValueError(
            "The resource contains invalid or out-of-range scores. "
            "No frozen CSV or hash has been written; inspect "
            "audit/lexicon_score_validation_failures.csv."
        )

    frozen_token_matchability_audit, frozen_token_matchability_summary = (
        write_lexicon_token_matchability_audit(
            frozen_lexicon,
            [shona_out, expanded_out],
            score_out,
            "frozen_lexicon",
        )
    )

    additions_df.to_csv(TAB / "validated_additions_v7_4.csv", index=False)

    # Multilingual enrichment can happen after evaluation without changing the frozen Shona form,
    # polarity or score. This template records what remains to be completed for the release resource.
    metadata_columns = ["candidate_id", "word", "researcher_meaning", "researcher_lexical_sentiment", "final_sentiment", "final_sentiment_basis", "final_score_minus9_to9"]
    metadata_template = validated_additions[metadata_columns].copy() if len(validated_additions) else pd.DataFrame(columns=metadata_columns)
    for column in ["English_final", "French_final", "Ciluba_final", "Zulu_final", "Afrikaans_final", "Sepedi_final", "Xhosa_final", "lexical_category_final", "metadata_checked_by"]:
        metadata_template[column] = ""
    metadata_template.to_csv(TAB / "validated_additions_v7_4_multilingual_metadata_template.csv", index=False)

    frozen_path = ROOT_OUT / "FrenchyShona_V2_v7_4_frozen.csv"
    frozen_lexicon.to_csv(frozen_path, index=False)
    freeze_record = {
        "notebook_version": NOTEBOOK_VERSION,
        "source_base_rows": int(len(master_lex_input)),
        "quarantined_inherited_score_rows": int(len(base_quarantine)),
        "base_rows": int(len(master_clean)),
        "source_base_lexicon_sha256": sha256_file(MASTER_LEX_PATH),
        "evaluation_ready_base_sha256": sha256_file(clean_base_path),
        "quarantine_file_sha256": sha256_file(
            AUD / "base_lexicon_quarantined_score_rows.csv"
        ),
        "statistical_shortlist": int(len(shortlist)),
        "researcher_review_complete": bool(researcher_review_complete),
        "validated_additions": int(len(additions_df)),
        "final_rows": int(len(frozen_lexicon)),
        "frozen_direct_single_token_unique_match_keys": int(
            frozen_token_matchability_summary["direct_single_token_unique_match_keys"]
        ),
        "frozen_multiword_entries_excluded_token_only": int(
            frozen_token_matchability_summary["multiword_entries_excluded"]
        ),
        "sha256": sha256_file(frozen_path),
        "build_text_label_sha256": BOUNDARY_BUILD_DIGEST,
        "validation_text_label_sha256": BOUNDARY_VALIDATION_DIGEST,
        "strict_test_text_label_sha256": BOUNDARY_TEST_DIGEST,
        "statistical_shortlist_sha256": sha256_file(TAB / "pmi_shortlist_v2.csv"),
        "candidate_recovery_table_sha256": sha256_file(TAB / "rag_candidate_recovery.csv"),
        "contextual_assessment_table_sha256": sha256_file(
            TAB / "rag_contextual_candidate_assessment_complete.csv"
        ),
        "candidate_evidence_comparison_sha256": sha256_file(TAB / "candidate_evidence_comparison.csv"),
        "contextual_assessment_model": RAG_MODEL,
        "candidate_recovery_prompt_version": RAG_RECOVERY_PROMPT_VERSION,
        "context_assessment_prompt_version": RAG_CONTEXT_PROMPT_VERSION,
        "final_decision_rule_version": FINAL_DECISION_RULE_VERSION,
        "contextual_statuses_allowed_for_corpus_association": sorted(
            CONTEXTUAL_STATUSES_ALLOWED_FOR_CORPUS_ASSOCIATION
        ),
        "single_researcher_review_file_sha256": sha256_file(REVIEW_PATH) if REVIEW_PATH else None,
        "single_researcher_decisions_sha256": sha256_file(REV / "single_researcher_decisions.csv") if (REV / "single_researcher_decisions.csv").exists() else None,
        "test_text_used_before_freeze_only_for_overlap_conflict_audit": True,
        "test_labels_used_for_candidate_selection": False,
        "test_labels_used_for_threshold_selection": False,
        "test_performance_computed_before_freeze": False,
        "validation_design": VALIDATION_DESIGN,
        "resource_status": "interim",
        "independent_native_speaker_validation": False,
        "inter_rater_agreement_applicable": False,
        "inter_rater_agreement_status": "not_applicable_single_researcher",
        "external_participant_data_used": False,
        "intended_supersession": (
            "Replace with the V6 multi-reviewer resource after ethics "
            "approval/exemption and independent review."
        ),
    }
    write_json(ROOT_OUT / "FrenchyShona_V2_interim_freeze_record.json", freeze_record)
    print(json_text(freeze_record))
else:
    print("FrenchyShona V2 interim is not frozen. Complete the single-researcher review first.")


In [ ]:

# Strict unseen lexicon evaluation and classical baselines. Runs only after the freeze.

lexicon_metrics = None
classical_metrics = None
lexicon_validation_predictions = None
lexicon_test_predictions = None

if frozen_lexicon is not None:
    # Partition integrity was enforced against the declared build snapshot before freeze.
    lex = frozen_lexicon.copy()
    score_eval = find_col_ci(lex, ["Score"])
    shona_eval_cols = [c for c in [find_col_ci(lex, ["Shona"]), find_col_ci(lex, ["expanded_shona"])] if c]

    score_lists = defaultdict(list)
    for _, row in lex.iterrows():
        try:
            score = float(row[score_eval])
        except Exception:
            continue
        row_forms = set()
        for column in shona_eval_cols:
            form_text = normalize_text(row.get(column, ""))
            # This evaluation is token based. Multiword entries remain available for later
            # phrase-aware evaluation but are not silently treated as single tokens here.
            if form_text and " " not in form_text:
                row_forms.add(norm_token(form_text))
        for form in row_forms:
            score_lists[form].append(score)
    if LEXICON_FORM_SCORE_AGGREGATION == "median":
        score_map = {word: float(np.median(scores)) for word, scores in score_lists.items()}
    elif LEXICON_FORM_SCORE_AGGREGATION == "mean":
        score_map = {word: float(np.mean(scores)) for word, scores in score_lists.items()}
    else:
        raise ValueError(f"Unsupported LEXICON_FORM_SCORE_AGGREGATION: {LEXICON_FORM_SCORE_AGGREGATION}")

    write_json(AUD / "lexicon_token_matching_rule.json", {
        "matching": "exact token match after project normalisation",
        "multiword_entries_included": False,
        "duplicate_form_score_aggregation": LEXICON_FORM_SCORE_AGGREGATION,
        "no_match_prediction": "NEU",
        "base_matchability_summary": "audit/base_lexicon_token_matchability_summary.json",
        "frozen_matchability_summary": "audit/frozen_lexicon_token_matchability_summary.json",
        "coverage_interpretation_note": (
            "Coverage is conditional on forms that can enter this exact token matcher. "
            "Multiword and normalisation-mismatched forms are reported separately in the audits."
        ),
    })

    conflict_rows = [
        {"word": word, "n_rows": len(scores), "scores": "|".join(map(str, sorted(set(scores))))}
        for word, scores in score_lists.items() if len(set(scores)) > 1
    ]
    pd.DataFrame(conflict_rows, columns=["word", "n_rows", "scores"]).to_csv(
        AUD / "lexicon_form_score_conflicts.csv", index=False
    )

    def tweet_features(text):
        tokens = tokenize_lexical_text(text)
        matched = [score_map[word] for word in tokens if word in score_map]
        if not matched:
            return {
                "match_count": 0, "token_count": len(tokens), "coverage_ratio": 0.0,
                "sum_score": 0.0, "avg_score": 0.0,
                "max_abs_score": 0.0, "score_std": 0.0
            }
        array = np.asarray(matched, dtype=float)
        return {
            "match_count": len(matched), "token_count": len(tokens),
            "coverage_ratio": len(matched) / max(1, len(tokens)),
            "sum_score": float(array.sum()),
            "avg_score": float(array.mean()), "max_abs_score": float(np.max(np.abs(array))),
            "score_std": float(array.std())
        }

    build_features = pd.DataFrame([tweet_features(text) for text in train_build[TEXT_COL]])
    build_features["gold"] = train_build["_label"].to_numpy()
    build_features["_norm_text"] = train_build["_norm_text"].to_numpy()
    validation_features = pd.DataFrame([tweet_features(text) for text in train_validation[TEXT_COL]])
    validation_features["gold"] = train_validation["_label"].to_numpy()
    validation_features["_norm_text"] = train_validation["_norm_text"].to_numpy()
    test_features = pd.DataFrame([tweet_features(text) for text in test_strict[TEXT_COL]])
    test_features["gold"] = test_strict["_label"].to_numpy()
    test_features["_norm_text"] = test_strict["_norm_text"].to_numpy()

    def sign_rule(values, match_count):
        values = np.asarray(values); matches = np.asarray(match_count)
        prediction = np.full(len(values), "NEU", dtype=object)
        prediction[(matches > 0) & (values < 0)] = "NEG"
        prediction[(matches > 0) & (values > 0)] = "POS"
        return prediction

    raw_test = sign_rule(test_features["sum_score"], test_features["match_count"])

    def predict_from_average(frame, negative_threshold, positive_threshold):
        scores = frame["avg_score"].to_numpy(); matches = frame["match_count"].to_numpy()
        prediction = np.full(len(frame), "NEU", dtype=object)
        prediction[(matches > 0) & (scores <= negative_threshold)] = "NEG"
        prediction[(matches > 0) & (scores >= positive_threshold)] = "POS"
        return prediction

    grid = np.arange(-9.0, 9.0001, THRESHOLD_GRID_STEP)
    threshold_rows = []
    best_key = None
    best_thresholds = None
    for negative_threshold in grid:
        for positive_threshold in grid:
            if negative_threshold >= positive_threshold:
                continue
            prediction = predict_from_average(validation_features, negative_threshold, positive_threshold)
            macro = f1_score(
                validation_features["gold"], prediction, labels=["NEG", "NEU", "POS"],
                average="macro", zero_division=0
            )
            accuracy = accuracy_score(validation_features["gold"], prediction)
            neutral_width = positive_threshold - negative_threshold
            distance_from_zero = abs(negative_threshold) + abs(positive_threshold)
            # Pre-declared tie break: macro F1, accuracy, thresholds closest to zero,
            # narrower neutral interval, then deterministic threshold values.
            key = (
                round(float(macro), 12), round(float(accuracy), 12),
                -round(float(distance_from_zero), 12), -round(float(neutral_width), 12),
                -round(float(negative_threshold), 12), -round(float(positive_threshold), 12),
            )
            threshold_rows.append({
                "negative_threshold": negative_threshold,
                "positive_threshold": positive_threshold,
                "macro_f1": macro, "accuracy": accuracy,
                "neutral_width": neutral_width,
                "distance_from_zero": distance_from_zero,
            })
            if best_key is None or key > best_key:
                best_key = key
                best_thresholds = (negative_threshold, positive_threshold)
    if best_thresholds is None:
        raise RuntimeError("No valid lexical threshold pair was available.")
    T_NEG, T_POS = best_thresholds
    write_json(EVAL / "validation_lexical_threshold_selection.json", {
        "selection_metric": "macro_f1",
        "secondary_metric": "accuracy",
        "tie_break": [
            "thresholds closest to zero",
            "narrower neutral interval",
            "deterministic threshold values",
        ],
        "grid_step": THRESHOLD_GRID_STEP,
        "selected_negative_threshold": float(T_NEG),
        "selected_positive_threshold": float(T_POS),
        "validation_macro_f1": float(best_key[0]),
        "validation_accuracy": float(best_key[1]),
        "no_match_prediction": "NEU",
    })
    pd.DataFrame(threshold_rows).sort_values(
        ["macro_f1", "accuracy", "distance_from_zero", "neutral_width"],
        ascending=[False, False, True, True]
    ).to_csv(EVAL / "validation_lexical_threshold_grid.csv", index=False)
    calibrated_validation = predict_from_average(validation_features, T_NEG, T_POS)
    calibrated_test = predict_from_average(test_features, T_NEG, T_POS)

    def metric_row(name, prediction):
        return {
            "model": name,
            "accuracy": accuracy_score(test_features["gold"], prediction),
            "macro_f1": f1_score(test_features["gold"], prediction, labels=["NEG", "NEU", "POS"], average="macro", zero_division=0),
            "weighted_f1": f1_score(test_features["gold"], prediction, labels=["NEG", "NEU", "POS"], average="weighted", zero_division=0),
        }

    lexicon_metrics = pd.DataFrame([
        metric_row("FrenchyShona V2 sign aggregate", raw_test),
        metric_row("FrenchyShona V2 calibrated average", calibrated_test),
    ])
    lexicon_metrics["coverage"] = float((test_features["match_count"] > 0).mean())
    lexicon_metrics["strict_test_rows"] = len(test_features)
    lexicon_metrics["validation_selected_neg_threshold"] = T_NEG
    lexicon_metrics["validation_selected_pos_threshold"] = T_POS
    lexicon_metrics.to_csv(EVAL / "lexicon_only_metrics_strict_test.csv", index=False)

    test_features["pred_raw"] = raw_test
    test_features["pred_calibrated"] = calibrated_test
    variant_diagnostic = {
        "raw_vs_calibrated_identical": bool(np.array_equal(raw_test, calibrated_test)),
        "raw_vs_calibrated_agreement": float(np.mean(raw_test == calibrated_test)),
        "validation_selected_negative_threshold": float(T_NEG),
        "validation_selected_positive_threshold": float(T_POS),
        "interpretation": (
            "Identical predictions are a valid possible result, but the selected thresholds "
            "and score distribution should be inspected."
            if np.array_equal(raw_test, calibrated_test)
            else "The validation-calibrated rule changes at least one strict-test prediction."
        ),
    }
    write_json(EVAL / "lexicon_variant_prediction_diagnostic.json", variant_diagnostic)
    if variant_diagnostic["raw_vs_calibrated_identical"]:
        print("NOTE: sign-based and calibrated lexicon predictions are identical on the strict test.")
    test_features.to_csv(EVAL / "strict_test_lexicon_predictions.csv", index=False)
    validation_features["pred_calibrated"] = calibrated_validation
    validation_features.to_csv(EVAL / "validation_lexicon_predictions.csv", index=False)
    lexicon_validation_predictions = validation_features
    lexicon_test_predictions = test_features

    majority_label = train_build["_label"].value_counts().idxmax()
    majority_test = np.full(len(test_features), majority_label, dtype=object)
    uncertainty = paired_bootstrap_macro_f1(
        test_features["gold"].to_numpy(), calibrated_test, majority_test,
        reps=BOOTSTRAP_REPS, seed=SEED
    )
    uncertainty.update({"comparison": "FrenchyShona V2 calibrated minus majority baseline", "bootstrap_reps": BOOTSTRAP_REPS})
    write_json(EVAL / "lexicon_bootstrap_uncertainty.json", uncertainty)

    precision, recall, f1_values, support = precision_recall_fscore_support(
        test_features["gold"], calibrated_test, labels=["NEG", "NEU", "POS"], zero_division=0
    )
    pd.DataFrame({
        "class": ["NEG", "NEU", "POS"], "precision": precision,
        "recall": recall, "f1": f1_values, "support": support
    }).to_csv(EVAL / "lexicon_only_class_metrics_strict_test.csv", index=False)

    if RUN_CLASSICAL_BASELINES:
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, max_df=0.95)
        X_train = vectorizer.fit_transform(train_build[TEXT_COL].astype(str))
        X_test = vectorizer.transform(test_strict[TEXT_COL].astype(str))
        baseline_models = {
            "TF-IDF Linear SVM": LinearSVC(class_weight="balanced", random_state=SEED),
            "TF-IDF Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
            "TF-IDF Multinomial Naive Bayes": MultinomialNB(),
        }
        baseline_rows = []
        prediction_frame = pd.DataFrame({"_norm_text": test_strict["_norm_text"], "gold": test_strict["_label"]})
        baseline_predictions = {}
        for name, model in baseline_models.items():
            start = time.perf_counter(); model.fit(X_train, train_build["_label"]); train_seconds = time.perf_counter() - start
            start = time.perf_counter(); prediction = model.predict(X_test); inference_seconds = time.perf_counter() - start
            baseline_predictions[name] = prediction
            baseline_rows.append({
                "model": name,
                "accuracy": accuracy_score(test_strict["_label"], prediction),
                "macro_f1": f1_score(test_strict["_label"], prediction, labels=["NEG", "NEU", "POS"], average="macro", zero_division=0),
                "weighted_f1": f1_score(test_strict["_label"], prediction, labels=["NEG", "NEU", "POS"], average="weighted", zero_division=0),
                "train_seconds": train_seconds, "strict_test_inference_seconds": inference_seconds,
            })
            prediction_frame["pred_" + re.sub(r"[^a-z0-9]+", "_", name.casefold()).strip("_")] = prediction
        classical_metrics = pd.DataFrame(baseline_rows)
        classical_metrics.to_csv(EVAL / "classical_baseline_metrics_strict_test.csv", index=False)
        prediction_frame.to_csv(EVAL / "classical_baseline_predictions_strict_test.csv", index=False)

        svm_prediction = baseline_predictions["TF-IDF Linear SVM"]
        svm_vs_lexicon = paired_bootstrap_macro_f1(
            test_strict["_label"].to_numpy(), svm_prediction, calibrated_test,
            reps=BOOTSTRAP_REPS, seed=SEED + 1
        )
        svm_vs_lexicon["comparison"] = "TF-IDF Linear SVM minus FrenchyShona V2 calibrated"
        write_json(EVAL / "svm_vs_lexicon_paired_bootstrap.json", svm_vs_lexicon)

    # Lexical evaluation figures and strict-test confusion matrix
    correlation_columns = ["match_count", "coverage_ratio", "sum_score", "avg_score", "max_abs_score", "score_std"]
    correlation = test_features[correlation_columns].corr()
    fig, ax = plt.subplots(figsize=(8, 7)); image = ax.imshow(correlation.to_numpy(), aspect="auto")
    ax.set_xticks(range(len(correlation_columns)), correlation_columns, rotation=45, ha="right")
    ax.set_yticks(range(len(correlation_columns)), correlation_columns)
    for i in range(len(correlation_columns)):
        for j in range(len(correlation_columns)):
            ax.text(j, i, f"{correlation.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(image, ax=ax); ax.set_title("Correlation of FrenchyShona-derived lexical features")
    plt.tight_layout(); save_current_figure("fig_5_9_lexical_feature_correlation.png")

    coverage_values = [int((test_features["match_count"] > 0).sum()), int((test_features["match_count"] == 0).sum())]
    plt.figure(figsize=(6, 4)); bars = plt.bar(["Matched", "No match"], coverage_values)
    plt.ylabel("Strict unseen test tweets"); plt.title("FrenchyShona V2 coverage on the strict unseen test")
    for bar, value in zip(bars, coverage_values):
        plt.text(bar.get_x() + bar.get_width()/2, value, f"{value:,}", ha="center", va="bottom")
    plt.tight_layout(); save_current_figure(["fig_5_10_lexicon_coverage.png", "fig_lexicon_coverage.png"])

    plot_frame = lexicon_metrics.set_index("model")[["accuracy", "macro_f1", "weighted_f1"]]
    axis = plot_frame.plot(kind="bar", figsize=(10, 5)); axis.set_ylim(0, 1); axis.set_ylabel("Score")
    axis.set_title("FrenchyShona V2 lexicon-only evaluation on the strict unseen test")
    axis.tick_params(axis="x", rotation=10); plt.tight_layout()
    save_current_figure(["fig_5_11_lexicon_only_evaluation.png", "fig_lexicon_only_results.png"])

    matrix = confusion_matrix(test_features["gold"], calibrated_test, labels=["NEG", "NEU", "POS"])
    fig, ax = plt.subplots(figsize=(5, 4.5)); image = ax.imshow(matrix)
    ax.set_xticks([0, 1, 2], ["NEG", "NEU", "POS"]); ax.set_yticks([0, 1, 2], ["NEG", "NEU", "POS"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Gold"); ax.set_title("FrenchyShona V2 calibrated confusion matrix")
    for i in range(3):
        for j in range(3): ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
    fig.colorbar(image, ax=ax); plt.tight_layout(); save_current_figure("fig_lexicon_only_confusion_strict_test.png")

    print(lexicon_metrics.to_string(index=False))
    if classical_metrics is not None: print("\n", classical_metrics.to_string(index=False))
else:
    for name, reason in [
        ("fig_5_9_lexical_feature_correlation.png", "Resource not frozen"),
        ("fig_5_10_lexicon_coverage.png", "Resource not frozen"),
        ("fig_5_11_lexicon_only_evaluation.png", "Resource not frozen"),
    ]:
        mark_figure_pending(name, reason)
    print("Held-out evaluation is skipped until the single-researcher review is complete and the interim resource is frozen.")


In [ ]:

# Optional contextual-model hybrid and paired bootstrap comparison

hybrid_summary = {"status": "not_run"}

probability_schema = pd.DataFrame([
    ["_norm_text", "normalize_text(text) used to align rows"],
    ["gold", "NEG, NEU or POS"],
    ["p_neg", "Contextual-model probability for NEG"],
    ["p_neu", "Contextual-model probability for NEU"],
    ["p_pos", "Contextual-model probability for POS"],
], columns=["column", "requirement"])
probability_schema.to_csv(MODEL / "model_probability_input_schema.csv", index=False)
write_json(MODEL / "model_probability_provenance_template.json", {
    "build_text_label_sha256": BOUNDARY_BUILD_DIGEST,
    "validation_text_label_sha256": BOUNDARY_VALIDATION_DIGEST,
    "strict_test_text_label_sha256": BOUNDARY_TEST_DIGEST,
    "model_name": "FILL_IN",
    "checkpoint_or_artifact": "FILL_IN",
    "training_data": "clean build split only",
    "class_order": ["NEG", "NEU", "POS"],
})

if frozen_lexicon is not None and DEV_MODEL_PROBS_PATH and TEST_MODEL_PROBS_PATH:
    if REQUIRE_MODEL_PROVENANCE and not MODEL_PROVENANCE_PATH:
        raise ValueError(
            "Model probability files were supplied without model_probability_provenance.json. "
            "The optional hybrid cannot be called clean unless the model was trained on the exact build split."
        )
    provenance = {}
    if MODEL_PROVENANCE_PATH:
        with open(MODEL_PROVENANCE_PATH, "r", encoding="utf-8") as handle:
            provenance = json.load(handle)
        expected_provenance = {
            "build_text_label_sha256": BOUNDARY_BUILD_DIGEST,
            "validation_text_label_sha256": BOUNDARY_VALIDATION_DIGEST,
            "strict_test_text_label_sha256": BOUNDARY_TEST_DIGEST,
        }
        mismatches = {
            key: {"expected": value, "observed": provenance.get(key)}
            for key, value in expected_provenance.items() if provenance.get(key) != value
        }
        if provenance.get("class_order") not in (None, ["NEG", "NEU", "POS"]):
            mismatches["class_order"] = {
                "expected": ["NEG", "NEU", "POS"],
                "observed": provenance.get("class_order"),
            }
        if mismatches:
            write_json(MODEL / "model_probability_provenance_mismatch.json", mismatches)
            raise ValueError(
                "Model probability provenance does not match the clean build/validation/test split."
            )

    validation_probs = pd.read_csv(DEV_MODEL_PROBS_PATH)
    test_probs = pd.read_csv(TEST_MODEL_PROBS_PATH)
    required = {"_norm_text", "gold", "p_neg", "p_neu", "p_pos"}
    expected_sets = {
        "validation": set(train_validation["_norm_text"]),
        "test": set(test_strict["_norm_text"]),
    }
    for name, frame in [("validation", validation_probs), ("test", test_probs)]:
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{name} probability file is missing: {sorted(missing)}")
        frame["_norm_text"] = frame["_norm_text"].map(normalize_text)
        if frame["_norm_text"].duplicated().any():
            raise ValueError(f"{name} probability file contains duplicate _norm_text rows")
        if set(frame["_norm_text"]) != expected_sets[name] or len(frame) != len(expected_sets[name]):
            raise ValueError(f"{name} probability rows do not exactly match the declared clean split")
        frame["gold"] = frame["gold"].map(norm_label)
        if not frame["gold"].isin(["NEG", "NEU", "POS"]).all():
            raise ValueError(f"{name} probability file contains invalid gold labels")
        probs = frame[["p_neg", "p_neu", "p_pos"]].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        if (
            not np.isfinite(probs).all()
            or (probs < -PROBABILITY_SUM_TOLERANCE).any()
            or (probs > 1 + PROBABILITY_SUM_TOLERANCE).any()
        ):
            raise ValueError(f"{name} probabilities must be finite and lie in [0,1]")
        row_sums = probs.sum(axis=1)
        if not np.allclose(row_sums, 1.0, atol=PROBABILITY_SUM_TOLERANCE, rtol=0):
            raise ValueError(f"{name} probability rows must sum to 1 within tolerance")
        frame[["p_neg", "p_neu", "p_pos"]] = probs

    dev = lexicon_validation_predictions.merge(validation_probs, on="_norm_text", how="inner", suffixes=("_lex", "_model"))
    test_joined = lexicon_test_predictions.merge(test_probs, on="_norm_text", how="inner", suffixes=("_lex", "_model"))
    if len(dev) != len(train_validation) or len(test_joined) != len(test_strict):
        raise ValueError("Model probabilities do not align one-to-one with the validation/test texts")
    if not (dev["gold_lex"].to_numpy() == dev["gold_model"].to_numpy()).all():
        raise ValueError("Validation probability gold labels do not match the reserved validation labels")
    if not (test_joined["gold_lex"].to_numpy() == test_joined["gold_model"].to_numpy()).all():
        raise ValueError("Test probability gold labels do not match the strict test labels")
    write_json(MODEL / "model_probability_provenance_audit.json", {
        "status": "pass",
        "provenance_file": str(MODEL_PROVENANCE_PATH) if MODEL_PROVENANCE_PATH else None,
        "build_text_label_sha256": BOUNDARY_BUILD_DIGEST,
        "validation_text_label_sha256": BOUNDARY_VALIDATION_DIGEST,
        "strict_test_text_label_sha256": BOUNDARY_TEST_DIGEST,
        "probability_sum_tolerance": PROBABILITY_SUM_TOLERANCE,
    })

    class_order = ["NEG", "NEU", "POS"]
    class_to_index = {label: i for i, label in enumerate(class_order)}

    def lexical_one_hot(prediction):
        output = np.zeros((len(prediction), 3), dtype=float)
        for i, label in enumerate(prediction): output[i, class_to_index[label]] = 1.0
        return output

    dev_model = dev[["p_neg", "p_neu", "p_pos"]].to_numpy(dtype=float)
    test_model = test_joined[["p_neg", "p_neu", "p_pos"]].to_numpy(dtype=float)
    dev_lex = lexical_one_hot(dev["pred_calibrated"].to_numpy())
    test_lex = lexical_one_hot(test_joined["pred_calibrated"].to_numpy())

    best = None
    alpha_rows = []
    for alpha_value in MODEL_ALPHA_GRID:
        probs = alpha_value * dev_model + (1 - alpha_value) * dev_lex
        prediction = np.asarray(class_order)[probs.argmax(axis=1)]
        score = f1_score(dev["gold_model"], prediction, labels=class_order, average="macro", zero_division=0)
        accuracy = accuracy_score(dev["gold_model"], prediction)
        alpha_rows.append({
            "alpha_contextual": alpha_value, "lexical_weight": 1 - alpha_value,
            "validation_macro_f1": score, "validation_accuracy": accuracy,
        })
        # Conservative tie break: prefer the larger contextual weight when macro F1 ties.
        candidate = (round(float(score), 12), alpha_value)
        if best is None or candidate > best:
            best = candidate
    pd.DataFrame(alpha_rows).to_csv(MODEL / "validation_hybrid_alpha_grid.csv", index=False)
    _, best_alpha = best

    hybrid_probs = best_alpha * test_model + (1 - best_alpha) * test_lex
    model_prediction = np.asarray(class_order)[test_model.argmax(axis=1)]
    hybrid_prediction = np.asarray(class_order)[hybrid_probs.argmax(axis=1)]
    gold = test_joined["gold_model"].to_numpy()

    comparison = paired_bootstrap_macro_f1(gold, hybrid_prediction, model_prediction, reps=BOOTSTRAP_REPS, seed=SEED + 2)
    hybrid_summary = {
        "status": "complete",
        "selected_alpha": float(best_alpha),
        "selected_lexical_weight": float(1 - best_alpha),
        "lexical_contribution_selected": bool(best_alpha < 1.0),
        "model_macro_f1": float(f1_score(gold, model_prediction, labels=class_order, average="macro", zero_division=0)),
        "hybrid_macro_f1": float(f1_score(gold, hybrid_prediction, labels=class_order, average="macro", zero_division=0)),
        **comparison,
    }
    if best_alpha == 1.0:
        hybrid_summary["interpretation"] = (
            "Validation selected alpha=1.0, so the final hybrid is the contextual model "
            "with no lexical contribution."
        )
    pd.DataFrame({
        "_norm_text": test_joined["_norm_text"], "gold": gold,
        "pred_model": model_prediction, "pred_hybrid": hybrid_prediction
    }).to_csv(MODEL / "strict_test_contextual_and_hybrid_predictions.csv", index=False)
    write_json(MODEL / "hybrid_paired_bootstrap.json", hybrid_summary)
    print(json_text(hybrid_summary))
else:
    print("Optional contextual-model hybrid not run; probability templates were created.")


In [ ]:

# Save run records, checksums, result workbooks and one results ZIP

config_record = {
    "seed": SEED,
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "minimum_document_support": MIN_DOC_SUPPORT,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "confidence_level": CI_LEVEL,
    "false_discovery_rate": FDR_ALPHA,
    "multiple_testing_test": "two-sided Fisher exact",
    "multiple_testing_correction": "Benjamini-Hochberg",
    "minimum_wilson_lower": MIN_WILSON_LOWER,
    "english_frequency_filter_active": bool(WORD_FREQ_AVAILABLE),
    "english_zipf_threshold": ENGLISH_ZIPF_THRESHOLD,
    "contextual_recovery_active": bool(USE_RAG_GPT),
    "complete_contextual_schema_required": True,
    "contextual_min_confidence": RAG_CONTEXT_MIN_CONFIDENCE,
    "final_decision_rule_version": FINAL_DECISION_RULE_VERSION,
    "contextual_statuses_allowed_for_corpus_association": sorted(
        CONTEXTUAL_STATUSES_ALLOWED_FOR_CORPUS_ASSOCIATION
    ),
    "contextual_assessment_model": RAG_MODEL,
    "contextual_recovery_min_support": RAG_RECOVERY_MIN_DOCUMENT_SUPPORT,
    "contextual_recovery_min_confidence": RAG_RECOVERY_MIN_CONFIDENCE,
    "contextual_recovery_batch_size": RAG_RECOVERY_BATCH_SIZE,
    "context_assessment_batch_size": RAG_CONTEXT_BATCH_SIZE,
    "candidate_recovery_prompt_version": RAG_RECOVERY_PROMPT_VERSION,
    "context_assessment_prompt_version": RAG_CONTEXT_PROMPT_VERSION,
    "contextual_evidence_data": "clean build split only",
    "hashtags": "marker removed, lexical content preserved",
    "expected_input_base_lexicon_rows": EXPECTED_INPUT_BASE_LEXICON_ROWS,
    "expected_quarantined_score_rows": EXPECTED_QUARANTINED_SCORE_ROWS,
    "expected_evaluation_ready_base_rows": EXPECTED_EVALUATION_BASE_LEXICON_ROWS,
    "base_score_integrity_criterion": "numeric, non-missing and within [-9, 9]",
    "exclude_label_conflicts": EXCLUDE_LABEL_CONFLICTS,
    "validation_design": VALIDATION_DESIGN,
    "interim_resource": INTERIM_RESOURCE,
    "reviewer_mode": "researcher_self_review_only",
    "required_reviewer_id": RESEARCHER_REVIEWER_ID,
    "minimum_researcher_reviewers": MIN_RESEARCHER_REVIEWERS,
    "minimum_valid_shona_share": MIN_VALID_SHONA_SHARE,
    "minimum_polarity_share": MIN_POLARITY_SHARE,
        "maximum_non_content_share": MAX_NON_CONTENT_SHARE,
    "require_complete_researcher_review": REQUIRE_COMPLETE_RESEARCHER_REVIEW,
        "require_researcher_meaning": REQUIRE_RESEARCHER_MEANING,
    "reject_identity_terms_in_interim": REJECT_IDENTITY_TERMS_IN_INTERIM,
    "inter_rater_agreement_applicable": False,
    "threshold_selection_data": "reserved validation split from the clean published training partition",
    "validation_fraction": VALIDATION_FRACTION,
    "test_text_used_before_freeze_only_for_overlap_conflict_audit": True,
    "test_labels_used_for_selection": False,
    "test_performance_computed_before_freeze": False,
    "clipped_lexicon_variant_reported_as_separate_system": False,
    "output_folder_reset_at_start": bool(RESET_OUTPUT_ROOT),
        "lexicon_form_score_aggregation": LEXICON_FORM_SCORE_AGGREGATION,
    "classical_training_data": "clean build split only",
    "require_model_probability_provenance": REQUIRE_MODEL_PROVENANCE,
}
write_json(ROOT_OUT / "run_config.json", config_record)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "openpyxl": openpyxl.__version__,
    "openai_sdk": getattr(openai_pkg, "__version__", "unknown"),
    "wordfreq_available": bool(WORD_FREQ_AVAILABLE),
}
write_json(ROOT_OUT / "environment_versions.json", environment)

run_summary = {
    "notebook_version": NOTEBOOK_VERSION,
    "published_train_rows": int(len(train_raw)),
    "published_test_rows": int(len(test_raw)),
    "strict_development_rows": int(len(train_strict)),
    "build_rows_used_for_lexicon": int(len(train_build)),
    "validation_rows_reserved_for_thresholds": int(len(train_validation)),
    "strict_unseen_test_rows": int(len(test_strict)),
    "source_base_lexicon_rows": int(len(master_lex_input)),
    "quarantined_inherited_score_rows": int(len(base_quarantine)),
    "base_lexicon_rows": int(len(master_lex)),
    "base_direct_single_token_unique_match_keys": int(
        base_token_matchability_summary["direct_single_token_unique_match_keys"]
    ),
    "candidate_tokens_missing_from_base": int(len(missing_candidates)),
    "deterministic_candidate_tokens": int(len(deterministic_candidates)),
    "contextual_recovery_pool": int(len(rag_recovery_pool)),
    "contextually_recovered_tokens": int(len(recovered_tokens)),
    "contextually_assessed_shortlist": int(len(rag_context_assessment)),
    "pmi_candidates_tested": int(len(pmi_all)),
    "statistical_shortlist": int(len(shortlist)),
    "single_researcher_review_supplied": bool(REVIEW_PATH),
    "single_researcher_review_complete": bool(researcher_review_complete),
    "provisional_keep_decisions": int(len(validated_additions)),
    "validated_additions_in_frozen_resource": (
        int(len(validated_additions)) if frozen_lexicon is not None else 0
    ),
    "frozen_lexicon_created": bool(frozen_lexicon is not None),
    "final_frozen_rows": int(len(frozen_lexicon)) if frozen_lexicon is not None else None,
    "frozen_direct_single_token_unique_match_keys": (
        int(frozen_token_matchability_summary["direct_single_token_unique_match_keys"])
        if frozen_lexicon is not None else None
    ),
    "optional_hybrid_status": hybrid_summary.get("status"),
    "inter_rater_agreement_applicable": False,
    "resource_status": "interim",
    "validation_design": VALIDATION_DESIGN,
    "independent_native_speaker_validation": False,
    "partition_boundary_assertions_passed": True,
        }
write_json(ROOT_OUT / "run_summary.json", run_summary)
# Consolidated results workbook. Detailed machine-readable audits remain in the output folder.
workbook_path = ROOT_OUT / "FrenchyShona_V2_results_tables.xlsx"
workbook_sources = [
    ("discovery_funnel", TAB / "candidate_discovery_funnel.csv"),
    ("full_funnel", TAB / "candidate_funnel_v2.csv"),
    ("rag_recovery", TAB / "rag_candidate_recovery.csv"),
    ("rag_metadata", TAB / "rag_run_metadata.csv"),
    ("candidate_filter", TAB / "candidate_filter_audit.csv"),
    ("document_support", TAB / "candidate_document_support_audit.csv"),
    ("pmi_candidates", TAB / "pmi_all_candidates_v2.csv"),
    ("pmi_shortlist", TAB / "pmi_shortlist_v2.csv"),
    ("discovery_ablation", TAB / "candidate_discovery_ablation.csv"),
    ("rag_context", TAB / "rag_contextual_candidate_assessment_complete.csv"),
    ("rag_context_metadata", TAB / "rag_context_run_metadata.csv"),
    ("evidence_comparison", TAB / "candidate_evidence_comparison.csv"),
    ("review_items", REV / "single_researcher_review_items.csv"),
    ("review_template", REV / "single_researcher_review_template.csv"),
    ("review_decisions", REV / "single_researcher_decisions.csv"),
    ("validated_additions", TAB / "validated_additions_v7_4.csv"),
    ("lexicon_results", EVAL / "lexicon_evaluation_results.csv"),
    ("classical_results", MODEL / "classical_model_results.csv"),
]
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    wrote_sheet = False
    for sheet_name, csv_path in workbook_sources:
        if csv_path.exists():
            frame = pd.read_csv(csv_path)
            frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            wrote_sheet = True
    if not wrote_sheet:
        pd.DataFrame({"status": ["No tabular outputs available"]}).to_excel(
            writer, sheet_name="status", index=False
        )

figure_manifest_frame = pd.DataFrame(figure_manifest)
if len(figure_manifest_frame):
    figure_manifest_frame = figure_manifest_frame.drop_duplicates(subset=["file"], keep="last")
else:
    figure_manifest_frame = pd.DataFrame(columns=["file", "status", "reason"])
figure_manifest_frame.to_csv(ROOT_OUT / "figure_manifest.csv", index=False)

hash_rows = []
for path in sorted(ROOT_OUT.rglob("*")):
    if path.is_file() and path.suffix.lower() != ".zip":
        hash_rows.append({
            "relative_path": str(path.relative_to(ROOT_OUT)),
            "sha256": sha256_file(path), "bytes": path.stat().st_size
        })
pd.DataFrame(hash_rows).to_csv(ROOT_OUT / "output_checksums.csv", index=False)

# Write the final archive beside the selected output folder. If that location is not
# writable, try the current working directory, /mnt/data, and the home directory.
zip_name = "FrenchyShona_V2_RAG_Complete_V7_4_results.zip"
zip_directories = [
    ROOT_OUT.parent,
    Path(OUTPUT_ROOT).expanduser().parent,
    Path.cwd(),
    Path("/mnt/data"),
    Path.home(),
]

ZIP_PATH = None
zip_errors = []
seen_directories = set()

for directory in zip_directories:
    try:
        directory = directory.expanduser()
        directory_key = str(directory.resolve(strict=False))
    except Exception:
        directory_key = str(directory)

    if directory_key in seen_directories:
        continue
    seen_directories.add(directory_key)

    temp_zip = None
    try:
        directory.mkdir(parents=True, exist_ok=True)
        candidate_zip = directory / zip_name
        temp_zip = directory / f".{zip_name}.tmp"
        if temp_zip.exists():
            temp_zip.unlink()

        with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in ROOT_OUT.rglob("*"):
                if not path.is_file():
                    continue
                rel = path.relative_to(ROOT_OUT)
                suffix = path.suffix.lower()
                # Keep the downloadable archive research-facing: tables, spreadsheets,
                # figures, review CSVs, frozen resources and evaluation/model CSVs.
                include = suffix in {".csv", ".xlsx", ".png"}
                if include:
                    archive.write(
                        path,
                        arcname=str(Path("FrenchyShona_V2_RAG_Complete_V7_4") / rel),
                    )

        # Reopen before rename so a truncated archive is never published as final.
        with zipfile.ZipFile(temp_zip, "r") as check_archive:
            bad_member = check_archive.testzip()
            if bad_member is not None:
                raise RuntimeError(f"ZIP integrity check failed at member: {bad_member}")
        os.replace(temp_zip, candidate_zip)
        ZIP_PATH = candidate_zip
        break
    except Exception as exc:
        if temp_zip is not None and temp_zip.exists():
            temp_zip.unlink()
        zip_errors.append(f"{directory}: {type(exc).__name__}: {exc}")

if ZIP_PATH is None:
    raise RuntimeError(
        "The analysis completed, but the results ZIP could not be written. "
        "Tried:\n- " + "\n- ".join(zip_errors)
    )

print("\nRUN COMPLETE")
print(json_text(run_summary))
print("\nOutput folder:", ROOT_OUT)
print("Results ZIP:", ZIP_PATH)
print("Blinded researcher-review package:", review_zip_path)
